# UJIAN TENGAH SEMESTER (UTS)
## PEMROGRAMAN SAINS DATA II (DS-2313)
**Institut Teknologi Sains Bandung (ITSB)**  
**Fakultas Digital, Desain dan Bisnis | Program Studi Sains Data**

| Atribut | Keterangan |
| :--- | :--- |
| **Mata Kuliah / Kode** | Pemrograman Sains Data II / DS-2313 (3 SKS: T = 2, P = 1) |
| **Dosen Pengampu** | Bakti Siregar, S.Si., M.Sc |
| **Semester / Tahun Akademik** | Semester 4 / TA Ganjil 2026/2027 |
| **Hari / Tanggal** | Rabu, 11 November 2026 |
| **Waktu** | $150 \text{ menit } (1 \times 3 \times 50')$ |
| **Sifat Ujian** | Kerjakan Mandiri dan Presentasikan di Video YouTube (Maksimum 10 Menit) |
| **Nama / NIM** | Fityanandra Athar Adyaksa - 52250059 |
| **Kelas** | Data Science |

## Persiapan Lingkungan dan Dataset Ujian
Sel berikut memuat pustaka yang dibutuhkan, mendefinisikan fungsi utilitas pengujian (`uji` dan `jalankan_uji`), serta membangkitkan dataset ujian `hd` dan tabel referensi `klinik` sesuai spesifikasi soal (**tidak mengubah logika persiapan soal**).

In [1]:
import re
import numpy as np
import pandas as pd

# Utilitas pengujian publik (uji & jalankan_uji)
_DAFTAR_UJI = []

def uji(kondisi, pesan="Uji gagal"):
    """Memeriksa satu asersi pengujian publik dan mencatat hasilnya."""
    if not kondisi:
        raise AssertionError(f"[GAGAL] {pesan}")
    return True

def jalankan_uji(nama_bagian, daftar_fungsi_uji):
    """Menjalankan sekumpulan uji publik untuk suatu bagian soal."""
    lulus = 0
    total = len(daftar_fungsi_uji)
    print(f"=== Menjalankan Uji Publik {nama_bagian} ({total} uji) ===")
    for idx, (deskripsi, fn) in enumerate(daftar_fungsi_uji, start=1):
        try:
            fn()
            lulus += 1
            print(f"  [LULUS {idx}/{total}] {deskripsi}")
        except Exception as e:
            print(f"  [GAGAL {idx}/{total}] {deskripsi} -> {type(e).__name__}: {e}")
            raise
    print(f"Hasil {nama_bagian}: {lulus}/{total} uji publik LULUS.\n")

# Pembuatan data ujian (JANGAN DIUBAH)
URL = 'https://raw.githubusercontent.com/kb22/Heart-Disease-Prediction/master/dataset.csv'
asli = pd.read_csv(URL)

rng = np.random.default_rng(2026)
hd = asli.copy()
hd.insert(0, 'id_pasien', [f'P{i:04d}' for i in range(1, len(hd) + 1)])
hd['kode_klinik'] = rng.choice(['KL01', 'KL02', 'KL03', 'KL04', 'KL05'], len(hd))
hd['sex'] = hd['sex'].map({1: 'Laki-laki', 0: 'Perempuan'})
hd.loc[rng.choice(len(hd), 12, replace=False), 'sex'] = 'L'          # penulisan tidak konsisten
hd.loc[rng.choice(len(hd), 6, replace=False), 'sex'] = ' perempuan ' # huruf & spasi tidak konsisten
hd.loc[rng.choice(len(hd), 18, replace=False), 'chol'] = 0           # nilai mustahil (kode hilang)
hd.loc[rng.choice(len(hd), 15, replace=False), 'trestbps'] = np.nan  # nilai hilang
hd.loc[rng.choice(len(hd), 2, replace=False), 'chol'] = 5640         # galat input
hd['age'] = hd['age'].astype(str) + ' th'                            # angka sebagai teks
hd = pd.concat([hd, hd.sample(6, random_state=3)], ignore_index=True) # duplikat

klinik = pd.DataFrame({
    'kode_klinik': ['KL01', 'KL02', 'KL03', 'KL04', 'KL06'],
    'kota': ['Bekasi', 'Bandung', 'Jakarta', 'Bogor', 'Depok'],
    'tipe': ['RS Swasta', 'RSUD', 'RS Swasta', 'Klinik Utama', 'RSUD']
})

HD_ASLI = hd.copy()  # untuk memeriksa bahwa fungsi tidak mengubah masukan
print('hd:', hd.shape, '| klinik:', klinik.shape)  # hd: (309, 16) | klinik: (5, 3)

hd: (309, 16) | klinik: (5, 3)


---
## BAGIAN A — KONSEP
**20 poin | A1–A8 @ 2 poin, A9 @ 4 poin | CPL-1 | CPMK-1, CPMK-2**

### Soal A1 — Broadcasting dan Memori Array NumPy *(2 poin | CPL-1 | CPMK-1 | Sub-CPMK-1)*
> **Pertanyaan:** Tuliskan aturan *broadcasting* NumPy sebagai algoritma (pseudocode) yang menerima dua *shape* dan mengembalikan *shape* hasil atau galat. Apa hasil `(4, 3)` dengan `(3,)` dan `(4, 3)` dengan `(4,)`? Bagaimana program dapat mendeteksi bahwa sebuah potongan array adalah *view*?

#### Jawaban A1:

**1. Algoritma (Pseudocode) Aturan *Broadcasting* NumPy**

NumPy membandingkan dimensi dua array secara elemen-per-elemen **mulai dari dimensi paling kanan (belakang) menuju ke kiri**. Jika jumlah dimensi (*ndim*) berbeda, *shape* yang lebih pendek disisipi (*prepend*) nilai `1` di sebelah kiri hingga panjang kedua *shape* sama.

```text
ALGORITMA HitungShapeBroadcast(shape_A, shape_B):
    INPUT : shape_A (tuple bilangan bulat), shape_B (tuple bilangan bulat)
    OUTPUT: shape_hasil (tuple bilangan bulat) atau GALAT (ValueError)

    1. n_A <- panjang(shape_A)
    2. n_B <- panjang(shape_B)
    3. n_maks <- MAKS(n_A, n_B)

    4. // Ratakan kanan (right-align) dengan menyisipkan 1 di sebelah kiri
       A_pad <- (1 sebanyak (n_maks - n_A) kali) digabung dengan shape_A
       B_pad <- (1 sebanyak (n_maks - n_B) kali) digabung dengan shape_B

    5. shape_hasil <- daftar kosong []

    6. // Periksa kompatibilitas dari dimensi paling kanan ke kiri (atau indeks 0..n_maks-1 setelah dipad)
       UNTUK i dari (n_maks - 1) turun ke 0 LAKUKAN:
           d_A <- A_pad[i]
           d_B <- B_pad[i]

           JIKA d_A == d_B MAKA:
               Masukkan d_A ke awal shape_hasil
           JIKA TIDAK, JIKA d_A == 1 MAKA:
               Masukkan d_B ke awal shape_hasil   // dimensi A diregangkan mengikuti B
           JIKA TIDAK, JIKA d_B == 1 MAKA:
               Masukkan d_A ke awal shape_hasil   // dimensi B diregangkan mengikuti A
           JIKA TIDAK:
               LEMPAR GALAT ValueError("Operands could not be broadcast together")

    7. KEMBALIKAN tuple(shape_hasil)
```

---

**2. Penelusuran Hasil *Broadcasting***

* **Kasus 1: `(4, 3)` dengan `(3,)`**
  1. Penyamaan panjang dimensi (*left-padding* dengan `1`):
     * `shape_A = (4, 3)`
     * `shape_B = (1, 3)`
  2. Pemeriksaan dari kanan ke kiri:
     * Dimensi ke-1 (kanan): `3` dan `3` $\rightarrow$ **cocok**, hasil dimensi = `3`.
     * Dimensi ke-0 (kiri): `4` dan `1` $\rightarrow$ **kompatibel** (salah satu bernilai `1`), diregangkan menjadi `4`.
  3. **Hasil akhir:** **`(4, 3)`** (berhasil).

* **Kasus 2: `(4, 3)` dengan `(4,)`**
  1. Penyamaan panjang dimensi (*left-padding* dengan `1`):
     * `shape_A = (4, 3)`
     * `shape_B = (1, 4)`
  2. Pemeriksaan dari kanan ke kiri:
     * Dimensi ke-1 (kanan): `3` dan `4` $\rightarrow$ **tidak cocok** ($3 \neq 4$ dan tidak ada yang bernilai `1`).
  3. **Hasil akhir:** **Galat (`ValueError: operands could not be broadcast together with shapes (4,3) (4,)`)**. Agar kompatibel, array kedua harus di-*reshape* secara eksplisit menjadi `(4, 1)`.

---

**3. Cara Program Mendeteksi Apakah Potongan Array adalah *View***

Sebuah *view* tidak memiliki penyangga memori (*data buffer*) sendiri, melainkan merujuk langsung pada memori array asal. Program dapat mendeteksinya melalui tiga cara utama di NumPy:

1. **Memeriksa atribut `.base`**:
   * Jika `potongan.base is not None` (atau secara spesifik `potongan.base is arr_asli`), maka `potongan` adalah **view** (objek meminjam memori dari objek di `.base`). Jika `potongan.base is None`, maka array tersebut memiliki datanya sendiri (*copy* atau array asli).
2. **Memeriksa bendera kepemilikan memori (`flags.owndata`)**:
   * `potongan.flags.owndata` bernilai `False` untuk *view* dan `True` untuk *copy*.
3. **Menggunakan fungsi `np.shares_memory(arr_asli, potongan)`**:
   * Mengembalikan `True` jika kedua array berbagi blok memori fisik yang tumpang tindih di RAM.

### Soal A2 — Algoritma *Hash Join* pada `merge` dan Bahaya Kunci Ganda *(2 poin | CPL-1 | CPMK-1 | Sub-CPMK-1)*
> **Pertanyaan:** Jelaskan algoritma *hash join* di balik `merge` beserta kompleksitasnya, dibandingkan *nested loop join*. Mengapa *join* dengan kunci ganda di tabel kanan berbahaya, dan bagaimana program mendeteksinya sebelum *join* dijalankan?

#### Jawaban A2:

**1. Algoritma *Hash Join* vs *Nested Loop Join* beserta Kompleksitasnya**

Misalkan tabel kiri $L$ memiliki $M$ baris dan tabel kanan $R$ memiliki $N$ baris:

* **Algoritma *Hash Join* (digunakan oleh `pd.merge`):**
  1. **Fase *Build* (Membangun Tabel Hash):** Memindai satu kali tabel yang lebih kecil (atau tabel kanan $R$) untuk menghitung nilai fungsi hash $h(k)$ dari setiap kunci join $k \in R$, lalu menyimpan pasangan `(kunci, indeks_baris_R)` ke dalam ember (*hash bucket*). Kompleksitas waktu rata-rata: $\mathcal{O}(N)$.
  2. **Fase *Probe* (Pencarian Pasangan):** Memindai tabel kiri $L$ baris demi baris, menghitung $h(k)$ untuk setiap kunci di $L$, dan mencarinya di tabel hash dengan waktu pencarian konstan $\mathcal{O}(1)$ rata-rata. Kompleksitas waktu rata-rata: $\mathcal{O}(M)$.
  * **Kompleksitas Total *Hash Join*:** Waktu rata-rata **$\mathcal{O}(M + N)$** dan memori tambahan **$\mathcal{O}(N)$** (untuk menyimpan *hash table*).
* **Perbandingan dengan *Nested Loop Join*:**
  * *Nested loop join* membandingkan setiap baris di $L$ terhadap seluruh baris di $R$ menggunakan dua perulangan bersarang.
  * **Kompleksitas *Nested Loop Join*:** Waktu **$\mathcal{O}(M \times N)$** (kuadratik) dengan memori tambahan $\mathcal{O}(1)$. Untuk $M, N = 10^5$, *nested loop* membutuhkan $10^{10}$ operasi (sangat lambat), sedangkan *hash join* hanya sekitar $2 \times 10^5$ operasi.

---

**2. Mengapa *Join* dengan Kunci Ganda di Tabel Kanan Berbahaya?**

Pada skenario pengayaan data (*left join* / *many-to-one*, misalnya menggabungkan tabel pasien dengan tabel referensi klinik):
1. **Ledakan Baris (*Row Multiplication / Fan-out*) Tanpa Pesan Galat:** Jika satu `kode_klinik` muncul $k$ kali di tabel referensi kanan, maka setiap pasien dari klinik tersebut akan digandakan menjadi $k$ baris pada tabel hasil *merge*. Pandas tidak melempar error secara *default*.
2. **Distorsi Statistik dan Agregasi Hilir (*Silent Data Corruption*):** Duplikasi baris yang tidak disadari akan menggelembungkan jumlah pasien (`count`), menggeser nilai rata-rata (`mean`), merusak distribusi target, dan menyebabkan **kebocoran data (*data leakage*)** apabila baris ganda dari pasien yang sama terpisah ke dalam *train set* dan *test set*.

---

**3. Cara Program Mendeteksinya Sebelum *Join* Dijalankan**

Program dapat menerapkan *defensive programming* sebelum atau saat memanggil `merge`:
1. **Pemeriksaan Eksplisit Sebelum Join:**
   ```python
   if kanan[kunci].duplicated().any():  # atau: if not kanan[kunci].is_unique:
       raise ValueError(f"Kunci kanan '{kunci}' mengandung duplikat!")
   ```
2. **Menggunakan Parameter `validate` pada `pd.merge`:**
   ```python
   hasil = pd.merge(kiri, kanan, on=kunci, how='left', validate='many_to_one')  # atau 'm:1'
   ```
   Parameter ini memerintahkan Pandas memverifikasi keunikan kunci di tabel kanan terlebih dahulu dan langsung melempar `pd.errors.MergeError` bila ditemukan kunci ganda.

### Soal A3 — Mekanisme Data Hilang (MCAR, MAR, MNAR) *(2 poin | CPL-1 | CPMK-1 | Sub-CPMK-2)*
> **Pertanyaan:** Jelaskan MCAR, MAR, dan MNAR dengan contoh rekam medis. Tuliskan algoritma sederhana untuk menguji secara otomatis apakah nilai hilang suatu kolom konsisten dengan MCAR. Mengapa MNAR tidak dapat diuji dari data?

#### Jawaban A3:

**1. Penjelasan MCAR, MAR, dan MNAR beserta Contoh Rekam Medis**

Misalkan $Y$ adalah variabel yang memiliki nilai hilang, $R \in \{0, 1\}$ adalah indikator kehilangan ($R=1$ jika $Y$ hilang), $X_{\text{obs}}$ adalah variabel lain yang teramati di dataset, dan $Y_{\text{mis}}$ adalah nilai $Y$ yang hilang itu sendiri:

1. **MCAR (*Missing Completely at Random*)** — $P(R = 1 \mid X_{\text{obs}}, Y_{\text{mis}}) = P(R = 1)$:
   * Peluang suatu nilai hilang murni acak dan **tidak bergantung** pada variabel teramati mana pun maupun pada nilai yang hilang itu sendiri.
   * *Contoh Rekam Medis:* Hasil kolesterol (`chol`) beberapa pasien hilang karena tabung sampel darah tidak sengaja terjatuh di laboratorium atau gangguan listrik sesaat pada alat pengukur tekanan darah (`trestbps`).
2. **MAR (*Missing at Random*)** — $P(R = 1 \mid X_{\text{obs}}, Y_{\text{mis}}) = P(R = 1 \mid X_{\text{obs}})$:
   * Peluang nilai hilang bergantung secara sistematis pada **variabel lain yang tercatat/teramati ($X_{\text{obs}}$)**, tetapi tidak bergantung pada nilai $Y_{\text{mis}}$ setelah $X_{\text{obs}}$ dikontrol.
   * *Contoh Rekam Medis:* Dokter lebih jarang memerintahkan tes kolesterol lengkap (`chol`) pada pasien berusia muda (`age < 30`) yang tidak memiliki keluhan nyeri dada (`cp = 0`). Di sini, hilangnya `chol` bergantung pada `age` dan `cp` yang nilainya tercatat lengkap di rekam medis.
3. **MNAR (*Missing Not at Random*)** — $P(R = 1 \mid X_{\text{obs}}, Y_{\text{mis}})$ masih bergantung pada $Y_{\text{mis}}$:
   * Peluang nilai hilang bergantung langsung pada **nilai yang tidak teramati itu sendiri** (atau faktor laten yang tidak diukur).
   * *Contoh Rekam Medis:* Pasien dengan depresi segmen ST (`oldpeak`) atau detak jantung latihan (`thalach`) yang sangat buruk gagal menyelesaikan uji *treadmill* karena kelelahan parah sehingga nilainya kosong; atau pasien perokok berat/obesitas ekstrem sengaja mengosongkan kuesioner berat badan karena enggan mengakuinya.

---

**2. Algoritma Sederhana Pengujian Otomatis Konsistensi MCAR**

Jika data berstatus MCAR, maka distribusi setiap kolom teramati $X_j$ tidak boleh berbeda secara signifikan antara kelompok baris yang $Y$-nya hilang ($R=1$) dan kelompok baris yang $Y$-nya lengkap ($R=0$).

```text
ALGORITMA UjiKonsistensiMCAR(df, kolom_target, alpha = 0.05):
    INPUT : DataFrame df, nama kolom_target yang memiliki NaN, taraf nyata alpha
    OUTPUT: Status ("Konsisten MCAR" atau "Tolak MCAR / Indikasi MAR") beserta daftar kolom terkait

    1. R <- indikator biner (1 jika df[kolom_target] bernilai NaN, 0 jika tidak)
    2. JIKA jumlah(R == 1) == 0 ATAU jumlah(R == 0) == 0 MAKA:
           KEMBALIKAN "Tidak dapat diuji (tidak ada variasi nilai hilang)"

    3. kolom_lain <- semua kolom di df selain kolom_target dan kolom ID
    4. alpha_koreksi <- alpha / panjang(kolom_lain)   // Koreksi Bonferroni untuk uji ganda
    5. kolom_signifikan <- daftar kosong []

    6. UNTUK SETIAP kol DI kolom_lain LAKUKAN:
           JIKA kol bertipe numerik MAKA:
               grup_hilang <- df.loc[R == 1, kol].dropna()
               grup_lengkap <- df.loc[R == 0, kol].dropna()
               // Bandingkan lokasi sebaran dengan Mann-Whitney U test atau Welch's t-test
               p_val <- UjiMannWhitneyU(grup_hilang, grup_lengkap)
           JIKA TIDAK (kol bertipe kategorikal) MAKA:
               tabel_kontingensi <- pd.crosstab(df[kol].fillna("MISSING"), R)
               // Uji independensi Chi-Square
               p_val <- UjiChiSquare(tabel_kontingensi)

           JIKA p_val < alpha_koreksi MAKA:
               Tambahkan (kol, p_val) ke kolom_signifikan

    7. JIKA kolom_signifikan TIDAK KOSONG MAKA:
           KEMBALIKAN ("Tolak MCAR (Indikasi MAR)", kolom_signifikan)
       JIKA TIDAK:
           KEMBALIKAN ("Konsisten dengan MCAR", [])
```

---

**3. Mengapa MNAR Tidak Dapat Diuji dari Data Semata?**

Untuk membuktikan atau menolak MNAR, kita harus menguji apakah $P(R = 1 \mid X_{\text{obs}}, Y_{\text{mis}}) \neq P(R = 1 \mid X_{\text{obs}})$, yang berarti membandingkan distribusi nilai $Y$ pada grup yang teramati ($R=0$) dengan distribusi nilai $Y$ pada grup yang **hilang** ($R=1$). Karena nilai $Y_{\text{mis}}$ pada grup $R=1$ **tidak pernah tercatat di dalam dataset**, informasi empiris untuk menghitung distribusi tersebut tidak ada. Tanpa data eksternal (seperti pelacakan ulang / audit rekam medis fisik terhadap pasien yang datanya kosong), ketergantungan pada nilai yang hilang bersifat *unidentifiable* dari data observasi saja.

### Soal A4 — Deteksi Kode Hilang dan Kebijakan Konsensus *(2 poin | CPL-1 | CPMK-1 | Sub-CPMK-2)*
> **Pertanyaan:** Pada Pertemuan 3, aturan *"lonjakan frekuensi di ujung distribusi"* dipakai untuk mendeteksi kode hilang. Jelaskan logikanya. Mengapa `Pregnancies = 0` tidak ditandai, sedangkan `Insulin = 0` ditandai? Mengapa pembersih otomatis memakai kebijakan konsensus ($\ge 2$ aturan) sebelum mengubah data?

#### Jawaban A4:

**1. Logika Aturan *"Lonjakan Frekuensi di Ujung Distribusi"* (*Boundary Spike*)**

Pada sistem pencatatan lama atau formulir klinis, nilai yang tidak diukur sering diisi dengan **nilai sentinel (kode hilang)** berupa angka khusus di luar rentang normal—paling sering `0`, `-1`, atau `999`. Secara statistik:
* Variabel kontinu alami (seperti kolesterol, tekanan darah, atau kadar insulin) memiliki fungsi kepadatan peluang yang **melandai secara mulus (*smooth tail*)** saat mendekati batas bawah maupun batas atas.
* Apabila seluruh nilai hilang dikodekan sebagai satu angka konstan (misalnya `0`), maka seluruh massa probabilitas dari data yang hilang akan menumpuk di satu titik ekstrem tersebut. Akibatnya, terbentuk **dua anomali struktural sekaligus**:
  1. **Lonjakan frekuensi (*frequency spike*):** Frekuensi nilai `0` jauh lebih tinggi daripada nilai-nilai kecil di dekatnya.
  2. **Celah distribusi (*distribution gap*):** Terdapat jarak kosong yang lebar (misalnya $> 0.5 \times \text{IQR}$) antara nilai `0` dengan nilai terukur terkecil berikutnya.

---

**2. Mengapa `Pregnancies = 0` Tidak Ditandai, Sedangkan `Insulin = 0` Ditandai?**

* **`Pregnancies = 0` (Nilai Sah / Valid):**
  1. *Kontinuitas Sebaran (Tanpa Celah):* `Pregnancies` adalah variabel cacah diskrit ($0, 1, 2, 3, \dots$). Nilai `0` bersebelahan langsung dengan `1, 2, 3` tanpa *gap* ekstrem terhadap $\text{IQR}$, dan penurunan frekuensinya dari $0 \to 1 \to 2$ mengikuti pola distribusi cacah alami (seperti Poisson/Negatif Binomial).
  2. *Makna Domain:* Pasien yang belum pernah hamil secara faktual memiliki jumlah kehamilan `0`.
* **`Insulin = 0` (Kode Hilang / Mustahil):**
  1. *Disrupsi & Celah Sebaran:* Setelah nilai `0`, nilai insulin positif terkecil berikutnya melompat jauh (misal $\ge 14\text{ }\mu\text{U/mL}$), menciptakan celah besar terhadap $\text{IQR}$ disertai lonjakan ratusan baris di angka `0`.
  2. *Makna Fisiologis:* Kadar insulin serum `0` mutlak pada manusia hidup secara medis mustahil; angka `0` tersebut adalah kode pengganti untuk pemeriksaan lab yang tidak dilakukan.

---

**3. Mengapa Pembersih Otomatis Memakai Kebijakan Konsensus ($\ge 2$ Aturan)?**

Mengubah nilai data asli menjadi `NaN` adalah operasi destruktif bila salah sasaran (*false positive*). Jika pembersih otomatis hanya mengandalkan **satu aturan tunggal**, risiko kesalahan sangat tinggi:
* Jika hanya memakai aturan *"frekuensi nilai 0 tinggi"*, maka variabel seperti **`oldpeak = 0`** (depresi ST saat istirahat, di mana pasien normal memang bernilai `0.0`) atau **`Pregnancies = 0`** akan ikut terhapus secara keliru.
* Sebaliknya, jika menggabungkan **konsensus $\ge 2$ aturan independen**—misalnya: **(Aturan 1)** kolom bersifat kontinu (banyak nilai unik), **(Aturan 2)** frekuensi di titik `0` melonjak dibanding tetangga, **DAN (Aturan 3)** terpisah oleh celah $> 0.5 \cdot \text{IQR}$ dari nilai positif terkecil berikutnya—maka `oldpeak = 0` aman (karena menyatu mulus dengan `0.1, 0.2, ...` tanpa celah), sedangkan `chol = 0` (yang melompat dari `0` langsung ke $> 120\text{ mg/dL}$) terdeteksi secara presisi.

### Soal A5 — Kontrak `fit/transform` pada *Transformer* Data *(2 poin | CPL-1 | CPMK-1 | Sub-CPMK-2)*
> **Pertanyaan:** Jelaskan kontrak `fit/transform`. Mengapa transformasi yang mempelajari statistik harus berupa kelas, bukan fungsi? Apa arti konvensi atribut berakhiran `_` dan mengapa `fit` mengembalikan `self`? Apa yang harus terjadi bila `transform` dipanggil sebelum `fit`?

#### Jawaban A5:

**1. Kontrak `fit/transform`**

Kontrak `fit/transform` (dipopulerkan oleh *scikit-learn*) memisahkan proses pra-pemrosesan data menjadi dua tahap yang tegas:
* **`fit(X, y=None)`**: Mempelajari parameter atau statistik distribusi **hanya dari data latih (`X_train`)**—seperti median, $\text{IQR}$, rata-rata, simpangan baku, atau himpunan kategori unik—lalu menyimpannya di dalam objek tanpa mengubah data.
* **`transform(X)`**: Menggunakan parameter yang telah disimpan dari tahap `fit` untuk mengubah data masukan (`X_train`, `X_val`, maupun `X_test`) secara konsisten dan mengembalikan objek data baru.

---

**2. Mengapa Harus Berupa Kelas (*Class*), Bukan Fungsi Biasa?**

Transformasi yang bergantung pada statistik data bersifat **berkeadaan (*stateful*)**:
* Jika ditulis sebagai fungsi biasa `skalakan(df)` yang langsung menghitung median dan $\text{IQR}$ dari argumen yang diberikan, maka saat fungsi tersebut dipanggil pada `X_test`, ia akan menghitung median dari `X_test` itu sendiri (menyebabkan **kebocoran data / *data leakage*** dan inkonsistensi skala ketika `X_test` hanya berisi 1 baris di lingkungan produksi).
* Jika fungsi dipaksa mengembalikan tuple `(df_tertransformasi, dict_parameter)`, pengguna harus mengelola dan meneruskan kamus parameter secara manual untuk setiap langkah pipeline.
* Dengan **kelas (OOP)**, parameter statistik dikapsulasi sebagai *state* internal objek (`self.median_`, `self.iqr_`), sehingga objek yang sudah di-`fit` pada data latih dapat langsung memanggil `.transform(X_test)` kapan saja dengan skala yang dijamin identik.

---

**3. Arti Konvensi Atribut Berakhiran `_` dan Mengapa `fit` Mengembalikan `self`**

* **Atribut berakhiran garis bawah tunggal `_` (misalnya `self.median_`, `self.iqr_`, `self.kategori_`)**:
  * Menandakan bahwa atribut tersebut adalah **atribut hasil pembelajaran (*fitted attribute*)** yang dihitung dari data saat `.fit()` dijalankan, untuk membedakannya secara jelas dari hiperparameter/konfigurasi awal yang diberikan pengguna saat inisialisasi `__init__()` (seperti `self.kolom`).
  * Keberadaan atribut berakhiran `_` juga menjadi penanda standar untuk memeriksa apakah objek sudah pernah di-`fit` (`hasattr(self, 'median_')`).
* **Mengapa `fit(X)` mengembalikan `self` (`return self`)**:
  * Memungkinkan **perantaian metode (*method chaining*)** yang ringkas dan ekspresif dalam satu baris, seperti:
    ```python
    X_latih_skala = SkalaRobustKu(kolom_num).fit(X_latih).transform(X_latih)
    ```

---

**4. Apa yang Harus Terjadi Bila `transform` Dipanggil Sebelum `fit`?**

Metode `transform(X)` **wajib menolak eksekusi dan langsung melempar eksepsi `RuntimeError`** (atau `NotFittedError` pada *scikit-learn*) dengan pesan yang jelas. Hal ini mencegah program berjalan dengan atribut yang belum terdefinisi (`AttributeError` yang membingungkan) atau menggunakan nilai *default* sembarang yang merusak validitas fitur.

### Soal A6 — *Gradient Descent* Adaptif dan Metrik Regresi *(2 poin | CPL-1 | CPMK-2 | Sub-CPMK-3)*
> **Pertanyaan:** Tuliskan pseudocode *gradient descent* untuk regresi linear yang (a) memiliki kriteria henti dan (b) mendeteksi divergensi serta memperbaikinya sendiri. Tuliskan rumus $R^2$, MAE, RMSE dan jelaskan mengapa $\text{RMSE} \ge \text{MAE}$.

#### Jawaban A6:

**1. Pseudocode *Gradient Descent* Regresi Linear dengan Kriteria Henti & Perbaikan Divergensi Otomatis**

Misalkan fungsi objektif adalah *Mean Squared Error* $J(w) = \frac{1}{2n} \|X w - y\|_2^2$ dengan gradien $\nabla J(w) = \frac{1}{n} X^T (X w - y)$:

```text
ALGORITMA GradientDescentRegresiLinear(X, y, lr = 0.1, tol = 1e-6, max_iter = 10000):
    INPUT : Matriks fitur X (n x p, sudah ditambah kolom 1 untuk intersep),
            vektor target y (n x 1), laju belajar awal lr, toleransi tol, batas iterasi max_iter
    OUTPUT: Vektor bobot optimal w (p x 1)

    1. n, p <- dimensi(X)
    2. w <- vektor nol berukuran (p x 1)
    3. loss_lama <- (1 / (2 * n)) * jumlah((X * w - y)^2)

    4. UNTUK iterasi = 1 SAMPAI max_iter LAKUKAN:
           galat <- (X * w) - y
           gradien <- (1 / n) * (transpose(X) * galat)

           // (a) Kriteria henti 1: Norma gradien sudah sangat mendekati nol
           JIKA norma_L2(gradien) < tol MAKA:
               BERHENTI (Konvergen)

           // Kandidat pembaruan bobot
           w_baru <- w - lr * gradien
           loss_baru <- (1 / (2 * n)) * jumlah((X * w_baru - y)^2)

           // (b) Deteksi divergensi (loss membesar atau menjadi NaN/Inf) & perbaikan mandiri
           JIKA tidak_berhingga(loss_baru) ATAU (loss_baru > loss_lama) MAKA:
               lr <- lr * 0.5                 // Pangkas learning rate separuhnya (backtracking)
               JIKA lr < 1e-15 MAKA:
                   BERHENTI                   // Mencegah loop tak hingga bila sudah di batas presisi
               LANJUT KE ITERASI BERIKUTNYA   // Tolak w_baru, pertahankan w lama
           
           // (a) Kriteria henti 2: Penurunan nilai loss sudah di bawah ambang toleransi
           JIKA |loss_lama - loss_baru| < tol MAKA:
               w <- w_baru
               BERHENTI (Konvergen)

           // Terima langkah pembaruan
           w <- w_baru
           loss_lama <- loss_baru

    5. KEMBALIKAN w
```

---

**2. Rumus Matematis $R^2$, MAE, dan RMSE**

Untuk target aktual $y = (y_1, \dots, y_n)$, prediksi $\hat{y} = (\hat{y}_1, \dots, \hat{y}_n)$, rata-rata $\bar{y} = \frac{1}{n}\sum_{i=1}^n y_i$, dan residual $e_i = y_i - \hat{y}_i$:

1. **Koefisien Determinasi ($R^2$):**
   $$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$$
2. ***Mean Absolute Error* (MAE):**
   $$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i| = \frac{1}{n} \sum_{i=1}^{n} |e_i|$$
3. ***Root Mean Squared Error* (RMSE):**
   $$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} e_i^2}$$

---

**3. Mengapa $\text{RMSE} \ge \text{MAE}$ Selalu Berlaku?**

Pandang himpunan galat mutlak $|e_1|, |e_2|, \dots, |e_n|$ sebagai peubah acak diskrit berseragam dengan nilai harapan $\mathbb{E}[|e|] = \text{MAE}$ dan momen kedua $\mathbb{E}[|e|^2] = \text{RMSE}^2$. Berdasarkan definisi varians yang selalu non-negatif ($\text{Var}(|e|) \ge 0$) maupun **Pertidaksamaan Jensen** pada fungsi konveks $g(u) = u^2$:

$$\text{Var}(|e|) = \mathbb{E}\left[|e|^2\right] - \left(\mathbb{E}[|e|]\right)^2 = \text{RMSE}^2 - \text{MAE}^2 \ge 0$$

Sehingga diperoleh hubungan eksak:

$$\text{RMSE} = \sqrt{\text{MAE}^2 + \text{Var}(|e|)} \ge \text{MAE}$$

* **Kesimpulan:** $\text{RMSE}$ selalu lebih besar atau sama dengan $\text{MAE}$. Kesamaan ($\text{RMSE} = \text{MAE}$) hanya terjadi jika $\text{Var}(|e|) = 0$, yaitu ketika seluruh residual memiliki besar mutlak yang identik ($|e_1| = |e_2| = \dots = |e_n|$). Semakin ekstrem *outlier* pada residual, semakin besar $\text{Var}(|e|)$ dan semakin jauh $\text{RMSE}$ melampaui $\text{MAE}$.

### Soal A7 — Stabilitas Numerik *Sigmoid*, *Log-Loss*, dan Interpretasi Regresi Logistik *(2 poin | CPL-1 | CPMK-2 | Sub-CPMK-3)*
> **Pertanyaan:** Mengapa `1/(1+np.exp(-z))` dan `np.log(p)` dapat gagal secara numerik? Tuliskan versi stabil *sigmoid* dan *log-loss*. Apa arti $\exp(\beta_j)$ pada regresi logistik?

#### Jawaban A7:

**1. Mengapa `1 / (1 + np.exp(-z))` dan `np.log(p)` Dapat Gagal Secara Numerik?**

* **Kegagalan `1 / (1 + np.exp(-z))` (*Floating-Point Overflow*):**
  * Tipe data `float64` (IEEE 754) memiliki batas nilai maksimum sekitar $1.797 \times 10^{308}$, sehingga fungsi eksponensial `np.exp(x)` akan mengalami **overflow menjadi `inf`** apabila argumennya $x > 709.78$.
  * Ketika logit $z$ bernilai negatif besar (misalnya $z = -800$), maka $-z = +800 > 709.78$. Akibatnya, `np.exp(-z)` menghasilkan `inf` dan memicu `RuntimeWarning: overflow encountered in exp` (meskipun secara matematis $1/(1+\infty) = 0$).
* **Kegagalan `np.log(p)` dan `np.log(1 - p)` (*Log of Zero & NaN*):**
  * Ketika $|z|$ cukup besar (misal $z > 37$ atau $z < -710$), probabilitas hasil sigmoid membulat secara *floating-point* menjadi tepat $p = 1.0$ atau $p = 0.0$.
  * Memanggil `np.log(0.0)` menghasilkan `-inf`. Lebih parah lagi, pada rumus *binary cross-entropy* $-y \log(p) - (1-y)\log(1-p)$, ketika $y = 0$ dan $p = 0$, suku pertama menghitung `0 * (-inf)` yang menghasilkan **`NaN`** dan merusak seluruh perhitungan gradien.

---

**2. Versi Stabil Secara Numerik untuk *Sigmoid* dan *Log-Loss***

* **a. *Sigmoid* Stabil (Memastikan Eksponen Selalu $\le 0$):**
  Gunakan identitas aljabar $\frac{1}{1 + e^{-z}} = \frac{e^z}{1 + e^z}$ untuk memisahkan kasus $z \ge 0$ dan $z < 0$ sehingga argumen `np.exp` tidak pernah positif:
  $$\sigma(z) = \begin{cases} \dfrac{1}{1 + e^{-z}}, & z \ge 0 \\[8pt] \dfrac{e^{z}}{1 + e^{z}}, & z < 0 \end{cases}$$

  ```python
  def sigmoid_stabil(z):
      z = np.asarray(z, dtype=float)
      hasil = np.empty_like(z, dtype=float)
      pos = (z >= 0)
      # Cabang z >= 0: eksponen -z <= 0 (bebas overflow tanpa perlu np.clip)
      hasil[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
      # Cabang z < 0 : eksponen z < 0 (bebas overflow & menghindari evaluasi eager np.where)
      exp_z = np.exp(z[~pos])
      hasil[~pos] = exp_z / (1.0 + exp_z)
      return hasil
  ```
  *(Catatan: Jika menggunakan `np.where(z >= 0, ...)`, karena NumPy mengevaluasi kedua argumen cabang secara *eager* sebelum memilih, kita dapat mengganti argumen eksponen menjadi `-np.abs(z)` atau memisahkan dengan masker boolean `pos = (z >= 0)` seperti di atas agar tidak memicu `RuntimeWarning` di balik layar tanpa perlu memotong nilai dengan `np.clip`).*

* **b. *Log-Loss* (*Binary Cross-Entropy*) Stabil:**
  1. **Jika masukan berupa probabilitas $p \in [0, 1]$** (gunakan *clipping* ke $[\varepsilon, 1 - \varepsilon]$ dengan $\varepsilon = 10^{-15}$):
     ```python
     def log_loss_dari_prob(y, p, eps=1e-15):
         p_aman = np.clip(p, eps, 1.0 - eps)
         return -np.mean(y * np.log(p_aman) + (1.0 - y) * np.log1p(-p_aman))
     ```
  2. **Jika masukan langsung berupa logit $z = Xw + b$** (substitusi analitik dengan `np.log1p` dan eksponen non-positif $-|z|$):
     $$\ell(y, z) = \max(z, 0) - y \cdot z + \ln\left(1 + e^{-|z|}\right)$$
     ```python
     def log_loss_dari_logit(y, z):
         return np.mean(np.maximum(z, 0.0) - y * z + np.log1p(np.exp(-np.abs(z))))
     ```

---

**3. Arti $\exp(\beta_j)$ pada Regresi Logistik**

Pada regresi logistik, model memprediksi *log-odds* (logit) kelas positif:
$$\ln\left(\frac{p}{1 - p}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_j x_j + \dots + \beta_k x_k$$

Jika fitur $x_j$ bertambah $1$ satuan menjadi $x_j + 1$ sementara seluruh fitur lainnya tetap konstan (*ceteris paribus*), maka rasio antara *odds* baru dan *odds* awal adalah:
$$\frac{\text{Odds}(x_j + 1)}{\text{Odds}(x_j)} = \frac{\exp(\beta_0 + \dots + \beta_j(x_j + 1) + \dots)}{\exp(\beta_0 + \dots + \beta_j x_j + \dots)} = \exp(\beta_j)$$

Dengan demikian, **$\exp(\beta_j)$ adalah *Odds Ratio* (OR)**, yaitu faktor pengali (*multiplicative factor*) terhadap *odds* ($\frac{p}{1-p}$) terjadinya kejadian positif ($Y=1$) untuk setiap kenaikan 1 unit pada variabel prediktor $x_j$:
* Jika $\beta_j > 0 \implies \exp(\beta_j) > 1$: kenaikan $x_j$ **meningkatkan** *odds* penyakit jantung sebesar $(\exp(\beta_j) - 1) \times 100\%$.
* Jika $\beta_j < 0 \implies 0 < \exp(\beta_j) < 1$: kenaikan $x_j$ **menurunkan** *odds* penyakit jantung.
* Jika $\beta_j = 0 \implies \exp(\beta_j) = 1$: fitur $x_j$ tidak mengubah *odds*.

### Soal A8 — Perhitungan Metrik Klasifikasi dan Kompleksitas ROC-AUC *(2 poin | CPL-1 | CPMK-2 | Sub-CPMK-4)*
> **Pertanyaan:** Dari *confusion matrix* $TN=90$, $FP=10$, $FN=30$, $TP=70$ hitung akurasi, presisi, recall, dan F1. Jelaskan cara menghitung ROC-AUC tanpa menggambar kurva beserta kompleksitasnya.

#### Jawaban A8:

**1. Perhitungan Manual Akurasi, Presisi, Recall, dan F1-Score**

Diketahui komponen *confusion matrix*:
* $TN = 90$, $FP = 10$, $FN = 30$, $TP = 70$
* Total sampel keseluruhan: $N = TN + FP + FN + TP = 90 + 10 + 30 + 70 = 200$

1. **Akurasi (*Accuracy*):**
   $$\text{Akurasi} = \frac{TP + TN}{N} = \frac{70 + 90}{200} = \frac{160}{200} = \mathbf{0.80} \text{ } (\mathbf{80\%})$$

2. **Presisi (*Precision* / *Positive Predictive Value*):**
   $$\text{Presisi} = \frac{TP}{TP + FP} = \frac{70}{70 + 10} = \frac{70}{80} = \mathbf{0.875} \text{ } (\mathbf{87.5\%})$$

3. **Recall (*Sensitivity* / *True Positive Rate*):**
   $$\text{Recall} = \frac{TP}{TP + FN} = \frac{70}{70 + 30} = \frac{70}{100} = \mathbf{0.70} \text{ } (\mathbf{70\%})$$

4. **F1-Score (Rata-rata Harmonik Presisi dan Recall):**
   $$\text{F1} = \frac{2 \times \text{Presisi} \times \text{Recall}}{\text{Presisi} + \text{Recall}} = \frac{2 \times 0.875 \times 0.70}{0.875 + 0.70} = \frac{1.225}{1.575} = \frac{7}{9} \approx \mathbf{0.7778} \text{ } (\mathbf{77.78\%})$$
   *(Atau langsung dengan rumus $\frac{2 TP}{2 TP + FP + FN} = \frac{2(70)}{140 + 10 + 30} = \frac{140}{180} = \frac{7}{9} \approx 0.7778$).*

---

**2. Cara Menghitung ROC-AUC Tanpa Menggambar Kurva beserta Kompleksitasnya**

Secara probabilistik, **ROC-AUC** sama dengan peluang bahwa sebuah sampel kelas positif ($y=1$) yang dipilih secara acak memiliki skor prediksi lebih tinggi daripada sampel kelas negatif ($y=0$) yang dipilih secara acak:
$$\text{AUC} = P(s_+ > s_-) + \frac{1}{2} P(s_+ = s_-)$$

Nilai ini ekuivalen dengan **statistik uji Mann-Whitney U (Wilcoxon Rank-Sum)** yang dihitung melalui tahapan berikut tanpa perlu menghitung titik-titik $(FPR, TPR)$ atau menggambar kurva:

1. **Pengurutan dan Pemberian Peringkat (*Ranking*):**
   * Gabungkan seluruh $N = n_+ + n_-$ skor prediksi, lalu urutkan secara menaik dari skor terkecil hingga terbesar.
   * Berikan peringkat $r_i \in \{1, 2, \dots, N\}$. Jika terdapat skor kembar (*ties*) dari indeks $a$ hingga $b$, berikan **peringkat rata-rata** $\frac{a + b}{2}$ kepada masing-masing elemen yang kembar tersebut.
2. **Penjumlahan Peringkat Kelas Positif ($R_+$):**
   * Hitung jumlah peringkat untuk seluruh sampel yang berlabel aktual positif ($y_i = 1$):
     $$R_+ = \sum_{i : y_i = 1} r_i$$
3. **Konversi ke Nilai AUC:**
   * Kurangi $R_+$ dengan jumlah peringkat minimum yang mungkin diperoleh $n_+$ sampel positif ($\frac{n_+(n_+ + 1)}{2}$), lalu bagi dengan total pasangan positif-negatif ($n_+ \times n_-$):
     $$\text{AUC} = \frac{R_+ - \dfrac{n_+(n_+ + 1)}{2}}{n_+ \cdot n_-}$$

* **Analisis Kompleksitas Waktu dan Ruang:**
  * Pendekatan perbandingan berpasangan langsung (*brute-force pairwise comparison*) membutuhkan $\mathcal{O}(n_+ \cdot n_-) = \mathcal{O}(N^2)$.
  * Dengan metode **peringkat Mann-Whitney**, langkah pengurutan $N$ skor membutuhkan waktu **$\mathcal{O}(N \log N)$**, sedangkan pemberian peringkat rata-rata dan penjumlahan $R_+$ hanya membutuhkan satu kali pemindaian linier **$\mathcal{O}(N)$**.
  * Jadi, kompleksitas waktu totalnya adalah **$\mathcal{O}(N \log N)$** dengan kompleksitas ruang **$\mathcal{O}(N)$**.

### Soal A9 — Konsep Pemrograman untuk Sains Data *(4 poin: @1 poin per bagian | CPL-1 | CPMK-1 | Sub-CPMK-1, Sub-CPMK-2)*
> **Pertanyaan:**  
> **(a)** Apa itu fungsi murni dan idempoten, dan mengapa keduanya penting untuk fungsi pembersihan data?  
> **(b)** Kapan memakai `raise` dan kapan memakai `assert`?  
> **(c)** Tuliskan kerangka *custom exception* `DataTidakValid` beserta cara menangkapnya.  
> **(d)** Apa itu *closure*? Berikan contoh dari evaluasi model.

#### Jawaban A9:

---

#### **(a) Fungsi Murni (*Pure Function*) dan Idempoten (*Idempotent Function*)**

* **Fungsi Murni (*Pure Function*):** Fungsi yang (1) keluarannya hanya ditentukan sepenuhnya oleh argumen masukannya (*deterministik*), dan (2) tidak menimbulkan **efek samping (*side effects*)** apa pun terhadap lingkungan luar—khususnya tidak memodifikasi DataFrame atau array masukan secara *in-place* dan tidak mengubah variabel global.
* **Fungsi Idempoten (*Idempotent Function*):** Fungsi yang apabila diterapkan berulang kali secara beruntun pada hasil keluarannya sendiri akan tetap menghasilkan keluaran yang sama persis seperti penerapan pertama:
  $$f(f(x)) = f(x)$$
* **Mengapa Keduanya Penting pada Pembersihan Data?**
  1. *Keamanan Eksplorasi di Notebook (Reprodusibilitas):* Di Jupyter Notebook, sel kode sering dieksekusi ulang secara tidak berurutan. Jika fungsi pembersihan tidak murni (mengubah `df` secara *in-place*), maka dataset mentah (`hd`) akan ikut terubah sehingga eksperimen berikutnya kehilangan kondisi awal.
  2. *Ketahanan Pipeline:* Sifat idempoten menjamin bahwa apabila `bersihkan(df)` tidak sengaja terpanggil dua kali dalam pipeline (`bersihkan(bersihkan(df))`), fungsi tidak akan gagal (misalnya error karena kolom `age` sudah berubah menjadi `float` dan tidak lagi memiliki metode `.str`), serta tidak akan menggeser nilai median atau menghapus baris tambahan.

---

#### **(b) Kapan Memakai `raise` dan Kapan Memakai `assert`?**

| Aspek | `raise` (Eksepsi Eksplisit) | `assert` (Asersi Internal) |
| :--- | :--- | :--- |
| **Tujuan Utama** | Menegakkan **kontrak API publik** dan menangani kesalahan masukan atau kondisi *runtime* yang tidak valid. | Memverifikasi **invarians internal (*sanity check* / *post-condition*)** selama pengembangan dan pengujian. |
| **Contoh Penggunaan** | Melempar `ValueError` saat kunci tabel kanan duplikat pada `gabung_aman`, atau melempar `RuntimeError` saat `transform()` dipanggil sebelum `fit()`. | Di baris akhir `bersihkan(df)`: `assert df_bersih.isna().sum().sum() == 0` untuk memastikan logika imputasi kita tidak menyisakan `NaN`. |
| **Perilaku di Produksi** | **Selalu dieksekusi** dalam kondisi apa pun; kita dapat memilih tipe eksepsi yang spesifik (`ValueError`, `KeyError`, `RuntimeError`). | **Dapat dinonaktifkan** oleh interpreter Python jika dijalankan dengan bendera optimasi (`python -O`), dan hanya melempar `AssertionError`. |

> **Aturan praktis:** Gunakan `raise` untuk memvalidasi argumen dari luar/pengguna; gunakan `assert` untuk memverifikasi kebenaran logika internal kode kita sendiri.

---

#### **(c) Kerangka *Custom Exception* `DataTidakValid` Beserta Cara Menangkapnya**

```python
class DataTidakValid(ValueError):
    """Eksepsi khusus ketika struktur atau nilai DataFrame melanggar aturan kualitas data."""
    def __init__(self, pesan, kolom=None, detail=None):
        self.kolom = kolom
        self.detail = detail
        info_kolom = f" [Kolom: '{kolom}']" if kolom else ""
        super().__init__(f"{pesan}{info_kolom}")


def validasi_kolesterol(df):
    """Contoh fungsi yang melempar DataTidakValid."""
    if 'chol' not in df.columns:
        raise DataTidakValid("Kolom wajib tidak ditemukan", kolom='chol')
    if (df['chol'] < 0).any():
        raise DataTidakValid("Ditemukan nilai kolesterol negatif", kolom='chol')
    return True


# Cara menangkap (handling) custom exception DataTidakValid:
try:
    df_uji = pd.DataFrame({'chol': [210, -5, 190]})
    validasi_kolesterol(df_uji)
except DataTidakValid as err:
    print(f"Tertangkap galat kualitas data -> {err} (pada kolom: {err.kolom})")
```

---

#### **(d) Pengertian *Closure* dan Contohnya pada Evaluasi Model**

* **Pengertian *Closure*:** *Closure* adalah sebuah fungsi dalam (*inner function*) yang dikembalikan oleh fungsi pembungkus (*enclosing/outer function*) dan **tetap mengingat (mengikat) variabel-variabel lokal dari ruang lingkup (*scope*) fungsi pembungkusnya**, meskipun fungsi pembungkus tersebut sudah selesai dieksekusi.
* **Contoh pada Evaluasi Model:**
  Ketika melakukan validasi silang (*cross-validation*), antarmuka evaluator biasanya hanya menerima fungsi dengan signature `skor(y_true, y_prob)`. Dengan *closure*, kita dapat menyuntikkan parameter konfigurasi (seperti ambang batas klasifikasi `ambang` atau bobot `beta` pada $F_\beta$-score) ke dalam fungsi evaluator:

```python
def buat_evaluator_fbeta(ambang=0.5, beta=2.0):
    """Outer function yang mengonfigurasi ambang dan bobot beta (mengembalikan closure)."""
    beta_kuadrat = beta ** 2  # variabel bebas (free variable) yang disimpan di dalam closure

    def hitung_skor(y_true, y_prob):
        """Inner function (closure) dengan signature standar (y_true, y_prob)."""
        y_true_arr = np.asarray(y_true)
        y_pred = (np.asarray(y_prob) >= ambang).astype(int)

        tp = np.sum((y_true_arr == 1) & (y_pred == 1))
        fp = np.sum((y_true_arr == 0) & (y_pred == 1))
        fn = np.sum((y_true_arr == 1) & (y_pred == 0))

        presisi = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        penyebut = (beta_kuadrat * presisi) + recall
        return ((1 + beta_kuadrat) * presisi * recall / penyebut) if penyebut > 0 else 0.0

    return hitung_skor


# Contoh pemakaian closure untuk skrining medis (menekankan Recall dengan beta=2, ambang=0.35):
skor_skrining = buat_evaluator_fbeta(ambang=0.35, beta=2.0)
# Fungsi skor_skrining kini cukup dipanggil dengan (y_true, y_prob) saja:
# nilai_f2 = skor_skrining(y_val, prob_val)
```

---
## BAGIAN B — PRAKTIKUM PEMROGRAMAN
**80 poin | B1–B6 | CPL-1 (30 poin), CPL-2 (50 poin) | CPMK-1, CPMK-2**

### Soal B1 — Detektor Masalah Kualitas Data Otomatis *(10 poin | CPL-2 | CPMK-1 | Sub-CPMK-2 | 9 uji publik)*

In [2]:
def _kolom_pengenal(df):
    """Mengidentifikasi kolom pengenal (ID) secara generik, tanpa nama kolom tertentu.

    Sebuah kolom dianggap pengenal bila namanya memuat token 'id' (mis. 'id', 'id_pasien',
    'pasien_id') atau bila kolom teks bernilai unik di setiap baris pada tabel berukuran
    cukup besar (>= 20 baris, agar tabel mungil tidak salah dikira berisi ID).
    """
    pola_nama = re.compile(r'(^|[_\s-])id($|[_\s-])', flags=re.IGNORECASE)
    pengenal = []
    for kol in df.columns:
        bertipe_teks = pd.api.types.is_string_dtype(df[kol]) or pd.api.types.is_object_dtype(df[kol])
        unik_semua = len(df) >= 20 and df[kol].nunique(dropna=True) == len(df)
        if pola_nama.search(str(kol)) or (bertipe_teks and unik_semua):
            pengenal.append(kol)
    return pengenal


def _pagar_ekstrem(seri):
    """Mengembalikan (q1, q3, iqr, batas_ekstrem) dengan batas_ekstrem = Q3 + 10 * IQR."""
    valid = seri.dropna()
    q1, q3 = float(valid.quantile(0.25)), float(valid.quantile(0.75))
    iqr = q3 - q1
    return q1, q3, iqr, q3 + 10.0 * iqr


def _mask_kategori_menyimpang(seri_teks):
    """Mask baris yang penulisannya menyimpang dari bentuk kanonik kategorinya.

    Dua penulisan dianggap satu kategori bila (i) hanya berbeda huruf besar/kecil atau spasi,
    atau (ii) salah satunya singkatan pendek (<= 2 huruf) yang menjadi awalan kategori lain
    (mis. 'l' untuk 'laki-laki'). Bentuk kanonik = penulisan mentah paling sering dalam satu keluarga.
    """
    normal = seri_teks.str.strip().str.lower()
    frek = normal.value_counts()
    peta_singkatan = {}
    for n in frek.index:
        if len(n) <= 2:
            calon = [m for m in frek.index if len(m) > len(n) and m.startswith(n)]
            if calon:
                peta_singkatan[n] = max(calon, key=lambda m: frek[m])
    keluarga = normal.map(lambda n: peta_singkatan.get(n, n))
    kanonik = seri_teks.groupby(keluarga.to_numpy()).agg(lambda g: g.value_counts().index[0])
    return seri_teks != keluarga.map(kanonik)


def deteksi_masalah(df, ref_klinik):
    """Mendeteksi 7 jenis masalah kualitas data secara otomatis tanpa mengubah df.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame masukan yang akan diperiksa kualitas datanya.
    ref_klinik : pd.DataFrame or None
        Tabel referensi klinik yang memuat kolom 'kode_klinik' (None = lewati KUNCI_YATIM).

    Returns
    -------
    pd.DataFrame
        Laporan dengan kolom ['jenis', 'kolom', 'jumlah', 'contoh'].
    """
    kolom_laporan = ['jenis', 'kolom', 'jumlah', 'contoh']
    if df is None or df.empty:
        return pd.DataFrame(columns=kolom_laporan)

    temuan = []
    pola_angka_satuan = re.compile(r'^\s*[+-]?\d+(?:[.,]\d+)?\s*[a-zA-Z%/]+\s*$')
    kolom_id = _kolom_pengenal(df)

    # 1. ANGKA_SEBAGAI_TEKS & 2. KATEGORI_TIDAK_KONSISTEN (hanya kolom teks non-ID)
    for kol in df.columns:
        if kol in kolom_id or kol == 'kode_klinik':
            continue
        s_non_na = df[kol].dropna()
        if s_non_na.empty:
            continue
        if not (pd.api.types.is_string_dtype(df[kol]) or pd.api.types.is_object_dtype(df[kol])):
            continue

        s_str = s_non_na.astype(str)
        mask_angka_teks = s_str.str.match(pola_angka_satuan)
        if mask_angka_teks.any():
            temuan.append({'jenis': 'ANGKA_SEBAGAI_TEKS', 'kolom': kol,
                           'jumlah': int(mask_angka_teks.sum()),
                           'contoh': str(s_str[mask_angka_teks].iloc[0])})
            continue

        mask_menyimpang = _mask_kategori_menyimpang(s_str)
        if mask_menyimpang.any():
            temuan.append({'jenis': 'KATEGORI_TIDAK_KONSISTEN', 'kolom': kol,
                           'jumlah': int(mask_menyimpang.sum()),
                           'contoh': str(s_str[mask_menyimpang].iloc[0])})

    # 3. HILANG, 4. NILAI_EKSTREM, 5. NOL_MUSTAHIL
    for kol in df.columns:
        s = df[kol]
        jml_hilang = int(s.isna().sum())
        if jml_hilang > 0:
            temuan.append({'jenis': 'HILANG', 'kolom': kol, 'jumlah': jml_hilang, 'contoh': np.nan})

        if not pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
            continue
        s_valid = s.dropna()
        if s_valid.empty:
            continue
        _, _, iqr, batas_ekstrem = _pagar_ekstrem(s_valid)
        if iqr <= 0:
            continue

        mask_ekstrem = s_valid > batas_ekstrem
        if mask_ekstrem.any():
            temuan.append({'jenis': 'NILAI_EKSTREM', 'kolom': kol,
                           'jumlah': int(mask_ekstrem.sum()),
                           'contoh': s_valid[mask_ekstrem].iloc[0].item()})

        # NOL_MUSTAHIL: kolom kontinu (> 10 nilai unik), nol melonjak dibanding nilai positif
        # terkecil DAN terpisah > 0.5 * IQR dari nilai positif terkecil itu
        if s_valid.nunique() > 10:
            jml_nol = int((s_valid == 0).sum())
            positif = s_valid[s_valid > 0]
            if jml_nol > 0 and not positif.empty:
                min_pos = float(positif.min())
                frek_tetangga = int((s_valid == min_pos).sum())
                if min_pos > 0.5 * iqr and jml_nol > frek_tetangga:
                    temuan.append({'jenis': 'NOL_MUSTAHIL', 'kolom': kol, 'jumlah': jml_nol, 'contoh': 0})

    # 6. DUPLIKAT (mengabaikan kolom ID)
    kolom_non_id = [c for c in df.columns if c not in kolom_id]
    if kolom_non_id:
        mask_duplikat = df.duplicated(subset=kolom_non_id, keep='first')
        if mask_duplikat.any():
            contoh = str(df.loc[mask_duplikat, kolom_id[0]].iloc[0]) if kolom_id else 'baris_ganda'
            temuan.append({'jenis': 'DUPLIKAT', 'kolom': 'semua',
                           'jumlah': int(mask_duplikat.sum()), 'contoh': contoh})

    # 7. KUNCI_YATIM (kode_klinik tanpa referensi di ref_klinik)
    if ref_klinik is not None and 'kode_klinik' in df.columns and 'kode_klinik' in ref_klinik.columns:
        kunci_sah = set(ref_klinik['kode_klinik'].dropna())
        s_klinik = df['kode_klinik'].dropna()
        mask_yatim = ~s_klinik.isin(kunci_sah)
        if mask_yatim.any():
            temuan.append({'jenis': 'KUNCI_YATIM', 'kolom': 'kode_klinik',
                           'jumlah': int(mask_yatim.sum()), 'contoh': str(s_klinik[mask_yatim].iloc[0])})

    return pd.DataFrame(temuan, columns=kolom_laporan)


laporan_masalah = deteksi_masalah(hd, klinik)
display(laporan_masalah)


,jenis,kolom,jumlah,contoh
0,ANGKA_SEBAGAI_TEKS,age,309,63 th
1,KATEGORI_TIDAK_KONSISTEN,sex,18,perempuan
2,HILANG,trestbps,16,NaN
3,NILAI_EKSTREM,chol,2,5640
4,NOL_MUSTAHIL,chol,18,0
5,DUPLIKAT,semua,6,P0246
6,KUNCI_YATIM,kode_klinik,56,KL05


In [3]:
# Sel Uji Publik B1 (9 Uji Publik)
def _uji_b1():
    hasil = deteksi_masalah(hd, klinik)
    uji_publik_b1 = [
        ("Struktur kolom sesuai ['jenis', 'kolom', 'jumlah', 'contoh']",
         lambda: uji(list(hasil.columns) == ['jenis', 'kolom', 'jumlah', 'contoh'])),
        ("Fungsi murni (tidak mengubah DataFrame masukan hd)",
         lambda: uji(hd.equals(HD_ASLI))),
        ("Mendeteksi ANGKA_SEBAGAI_TEKS pada 'age' (309 baris)",
         lambda: uji(int(hasil.query("jenis == 'ANGKA_SEBAGAI_TEKS' and kolom == 'age'")['jumlah'].iloc[0]) == 309)),
        ("Mendeteksi KATEGORI_TIDAK_KONSISTEN pada 'sex' (18 baris)",
         lambda: uji(int(hasil.query("jenis == 'KATEGORI_TIDAK_KONSISTEN' and kolom == 'sex'")['jumlah'].iloc[0]) == 18)),
        ("Mendeteksi NOL_MUSTAHIL pada 'chol' (18 baris) dan mengabaikan 'oldpeak'",
         lambda: uji(list(hasil.query("jenis == 'NOL_MUSTAHIL'")['kolom']) == ['chol']
                     and int(hasil.query("jenis == 'NOL_MUSTAHIL'")['jumlah'].iloc[0]) == 18)),
        ("Mendeteksi HILANG pada 'trestbps' (16 baris)",
         lambda: uji(int(hasil.query("jenis == 'HILANG' and kolom == 'trestbps'")['jumlah'].iloc[0]) == 16)),
        ("Mendeteksi NILAI_EKSTREM pada 'chol' (2 baris)",
         lambda: uji(int(hasil.query("jenis == 'NILAI_EKSTREM' and kolom == 'chol'")['jumlah'].iloc[0]) == 2)),
        ("Mendeteksi DUPLIKAT dengan mengabaikan ID (6 baris)",
         lambda: uji(int(hasil.query("jenis == 'DUPLIKAT'")['jumlah'].iloc[0]) == 6)),
        ("Mendeteksi KUNCI_YATIM pada 'kode_klinik' (56 baris) & tahan DataFrame kosong",
         lambda: uji(int(hasil.query("jenis == 'KUNCI_YATIM'")['jumlah'].iloc[0]) == 56
                     and deteksi_masalah(pd.DataFrame(), klinik).empty)),
    ]
    jalankan_uji("B1 - Detektor Masalah Kualitas Data", uji_publik_b1)

_uji_b1()

=== Menjalankan Uji Publik B1 - Detektor Masalah Kualitas Data (9 uji) ===
  [LULUS 1/9] Struktur kolom sesuai ['jenis', 'kolom', 'jumlah', 'contoh']
  [LULUS 2/9] Fungsi murni (tidak mengubah DataFrame masukan hd)
  [LULUS 3/9] Mendeteksi ANGKA_SEBAGAI_TEKS pada 'age' (309 baris)
  [LULUS 4/9] Mendeteksi KATEGORI_TIDAK_KONSISTEN pada 'sex' (18 baris)
  [LULUS 5/9] Mendeteksi NOL_MUSTAHIL pada 'chol' (18 baris) dan mengabaikan 'oldpeak'
  [LULUS 6/9] Mendeteksi HILANG pada 'trestbps' (16 baris)
  [LULUS 7/9] Mendeteksi NILAI_EKSTREM pada 'chol' (2 baris)
  [LULUS 8/9] Mendeteksi DUPLIKAT dengan mengabaikan ID (6 baris)
  [LULUS 9/9] Mendeteksi KUNCI_YATIM pada 'kode_klinik' (56 baris) & tahan DataFrame kosong
Hasil B1 - Detektor Masalah Kualitas Data: 9/9 uji publik LULUS.



In [4]:
# Uji tambahan B1 (kasus tepi buatan sendiri; di luar 9 uji publik soal)
def _uji_b1_tepi():
    kota = pd.DataFrame({'kota': ['Bandung', 'Bogor', 'Bekasi', 'Depok'] * 10, 'x': range(40)})
    satu_baris = deteksi_masalah(hd.head(1), klinik)
    uji_tepi = [
        ("Kota berawalan sama (Bandung/Bogor/Bekasi) TIDAK ditandai tidak konsisten",
         lambda: uji('KATEGORI_TIDAK_KONSISTEN' not in set(deteksi_masalah(kota, klinik)['jenis']))),
        ("Tabel 1 baris tetap mendeteksi ANGKA_SEBAGAI_TEKS pada 'age'",
         lambda: uji(((satu_baris['jenis'] == 'ANGKA_SEBAGAI_TEKS') & (satu_baris['kolom'] == 'age')).any())),
        ("Tanpa kolom kode_klinik dan ref_klinik=None tidak menimbulkan galat",
         lambda: uji(isinstance(deteksi_masalah(hd.drop(columns='kode_klinik'), None), pd.DataFrame))),
    ]
    jalankan_uji("B1 - uji tambahan kasus tepi", uji_tepi)

_uji_b1_tepi()

=== Menjalankan Uji Publik B1 - uji tambahan kasus tepi (3 uji) ===
  [LULUS 1/3] Kota berawalan sama (Bandung/Bogor/Bekasi) TIDAK ditandai tidak konsisten
  [LULUS 2/3] Tabel 1 baris tetap mendeteksi ANGKA_SEBAGAI_TEKS pada 'age'
  [LULUS 3/3] Tanpa kolom kode_klinik dan ref_klinik=None tidak menimbulkan galat
Hasil B1 - uji tambahan kasus tepi: 3/3 uji publik LULUS.



#### Interpretasi B1:
Hasil `deteksi_masalah(hd, klinik)` pada `hd` ($309 \times 16$) menemukan **7 jenis masalah** yang semuanya sesuai rancangan data ujian:
1. **`ANGKA_SEBAGAI_TEKS` (`age`, 309 baris):** seluruh usia berbentuk teks bersatuan (contoh `'63 th'`) sehingga harus diekstrak menjadi angka sebelum dipakai.
2. **`KATEGORI_TIDAK_KONSISTEN` (`sex`, 18 baris):** 12 baris `'L'` + 6 baris `' perempuan '`, menyimpang dari bentuk kanonik `'Laki-laki'` (201) dan `'Perempuan'` (90).
3. **`NOL_MUSTAHIL` (`chol`, 18 baris):** nilai positif terkecil `chol` adalah 126 mg/dL, terpaut jauh dari 0 (celah 126 > ½·IQR = 35,5) dan frekuensi nol (18) jauh melampaui frekuensi nilai terkecil berikutnya (1 baris). `oldpeak = 0` (102 baris) **tidak** ditandai: nilai positif terkecilnya 0,1 hanya terpaut 0,1 ≤ ½·IQR = 0,8, jadi nol itu menyatu dengan sebaran dan sah secara klinis (tidak ada depresi ST). Jadi frekuensi tinggi saja tidak cukup; syarat celah juga harus terpenuhi.
4. **`HILANG` (`trestbps`, 16 baris):** 15 NaN hasil penyuntikan + 1 NaN yang ikut tergandakan pada baris duplikat.
5. **`NILAI_EKSTREM` (`chol`, 2 baris):** dua nilai `5640` melampaui $Q_3 + 10\cdot\text{IQR} = 985$, jelas galat input.
6. **`DUPLIKAT` (6 baris):** terdeteksi dengan mengabaikan kolom pengenal.
7. **`KUNCI_YATIM` (`kode_klinik`, 56 baris):** seluruhnya `KL05`, kode yang tidak ada pada tabel `klinik`.

**Catatan ketahanan:** deteksi kategori memakai kunci ternormalisasi (huruf kecil, tanpa spasi) dan singkatan ≤ 2 huruf, bukan sekadar huruf awal, supaya kolom seperti `kota` (Bandung/Bogor/Bekasi) tidak salah ditandai; kolom ID dikenali dari nama/keunikan sehingga tabel mungil tetap terperiksa. Keduanya diuji pada sel uji tambahan di atas.

---
### Soal B2 — Wrangling yang Dapat Diuji *(15 poin | CPL-2 | CPMK-1 | Sub-CPMK-1 | 6 uji publik)*

In [5]:
def seragamkan(df):
    """Menyeragamkan kolom 'age' menjadi numerik dan 'sex' menjadi {'Laki-laki', 'Perempuan'}.

    Fungsi murni: tidak memodifikasi DataFrame masukan.
    """
    if df is None or df.empty:
        return df.copy() if df is not None else pd.DataFrame()

    df_out = df.copy()

    # (a.1) Ekstrak angka dari 'age' dan buang satuan teks apa pun menggunakan regex
    if 'age' in df_out.columns and not pd.api.types.is_numeric_dtype(df_out['age']):
        ekstrak_angka = (
            df_out['age']
            .astype(str)
            .str.extract(r'([+-]?\d+(?:\.\d+)?)', expand=False)
        )
        seri_num = pd.to_numeric(ekstrak_angka, errors='coerce')
        if seri_num.notna().all() and (seri_num % 1 == 0).all():
            df_out['age'] = seri_num.astype(int)
        else:
            df_out['age'] = seri_num.astype(float)

    # (a.2) Seragamkan 'sex' memakai peta otomatis dari kunci kanonik & aturan huruf awal
    if 'sex' in df_out.columns:
        kunci_kanonik = ('Laki-laki', 'Perempuan')
        aturan_huruf_awal = {k[0].lower(): k for k in kunci_kanonik}
        nilai_unik = df_out['sex'].dropna().unique()
        peta_sex = {
            val: aturan_huruf_awal.get(str(val).strip()[:1].lower(), val)
            for val in nilai_unik
        }
        df_out['sex'] = df_out['sex'].map(
            lambda v: peta_sex.get(v, v) if pd.notna(v) else v
        )

    return df_out


def gabung_aman(kiri, kanan, kunci):
    """Melakukan left join many-to-one yang tervalidasi, mencetak kunci yatim, dan menjaga jumlah baris."""
    if kunci not in kiri.columns or kunci not in kanan.columns:
        raise ValueError(f"Kolom kunci '{kunci}' harus tersedia pada kedua tabel.")

    # Validasi many-to-one sebelum join: kunci di tabel kanan wajib unik
    if kanan[kunci].duplicated().any():
        kunci_ganda = kanan.loc[kanan[kunci].duplicated(), kunci].unique().tolist()
        raise ValueError(f"Tabel kanan memiliki kunci ganda pada kolom '{kunci}': {kunci_ganda}")

    # Deteksi dan cetak kunci yatim
    kunci_yatim = sorted(set(kiri[kunci].dropna()) - set(kanan[kunci].dropna()))
    print(f"Kunci yatim terdeteksi pada '{kunci}' ({len(kunci_yatim)} kode): {kunci_yatim}")

    hasil = pd.merge(kiri.copy(), kanan.copy(), on=kunci, how='left', validate='many_to_one')
    assert len(hasil) == len(kiri), "Jumlah baris hasil join tidak sama dengan tabel kiri!"
    return hasil


# (c) Satu rantai method: rata-rata chol valid (100-700) dan persentase target == 1 per kota x sex
ringkasan_kota_sex = (
    gabung_aman(seragamkan(hd), klinik, 'kode_klinik')
    .query('100 <= chol <= 700')
    .groupby(['kota', 'sex'], as_index=True)
    .agg(
        rata_rata_chol=('chol', 'mean'),
        persen_target_1=('target', lambda s: s.mean() * 100.0),
    )
    .round(2)
)
display(ringkasan_kota_sex)

Kunci yatim terdeteksi pada 'kode_klinik' (1 kode): ['KL05']


rata_rata_chol  persen_target_1
kota    sex                                       
Bandung Laki-laki          256.48            40.48
        Perempuan          294.12            76.47
Bekasi  Laki-laki          225.35            50.00
        Perempuan          269.85            80.00
Bogor   Laki-laki          225.29            46.34
        Perempuan          240.55            63.64
Jakarta Laki-laki          250.82            52.27
        Perempuan          239.94            81.25

In [6]:
# Sel Uji Publik B2 (6 Uji Publik)
def _uji_b2():
    df_seragam = seragamkan(hd)
    df_gabung = gabung_aman(df_seragam, klinik, 'kode_klinik')
    kanan_rusak = pd.concat([klinik, klinik.iloc[[0]]], ignore_index=True)

    def _cek_value_error_kunci_ganda():
        try:
            gabung_aman(df_seragam, kanan_rusak, 'kode_klinik')
            return False
        except ValueError:
            return True

    uji_publik_b2 = [
        ("seragamkan bersifat murni (tidak mengubah hd)",
         lambda: uji(hd.equals(HD_ASLI))),
        ("Kolom 'age' hasil seragamkan bertipe numerik dan berada di rentang wajar (20-100)",
         lambda: uji(pd.api.types.is_numeric_dtype(df_seragam['age'])
                     and df_seragam['age'].between(20, 100).all())),
        ("Kolom 'sex' hasil seragamkan tepat bernilai {'Laki-laki', 'Perempuan'}",
         lambda: uji(set(df_seragam['sex'].dropna().unique()) == {'Laki-laki', 'Perempuan'})),
        ("gabung_aman mempertahankan jumlah baris tabel kiri (309 baris)",
         lambda: uji(len(df_gabung) == len(hd))),
        ("gabung_aman melempar ValueError saat tabel kanan memiliki kunci ganda",
         lambda: uji(_cek_value_error_kunci_ganda())),
        ("Tabel agregasi rantai method memiliki indeks (kota, sex) dan 8 kombinasi baris",
         lambda: uji(ringkasan_kota_sex.shape == (8, 2)
                     and list(ringkasan_kota_sex.index.names) == ['kota', 'sex'])),
    ]
    jalankan_uji("B2 - Wrangling yang Dapat Diuji", uji_publik_b2)

_uji_b2()

Kunci yatim terdeteksi pada 'kode_klinik' (1 kode): ['KL05']
=== Menjalankan Uji Publik B2 - Wrangling yang Dapat Diuji (6 uji) ===
  [LULUS 1/6] seragamkan bersifat murni (tidak mengubah hd)
  [LULUS 2/6] Kolom 'age' hasil seragamkan bertipe numerik dan berada di rentang wajar (20-100)
  [LULUS 3/6] Kolom 'sex' hasil seragamkan tepat bernilai {'Laki-laki', 'Perempuan'}
  [LULUS 4/6] gabung_aman mempertahankan jumlah baris tabel kiri (309 baris)
  [LULUS 5/6] gabung_aman melempar ValueError saat tabel kanan memiliki kunci ganda
  [LULUS 6/6] Tabel agregasi rantai method memiliki indeks (kota, sex) dan 8 kombinasi baris
Hasil B2 - Wrangling yang Dapat Diuji: 6/6 uji publik LULUS.



#### Interpretasi B2:
1. **`seragamkan`:** `age` kini bertipe `int64` (rentang 29–77 tahun) tanpa baris hilang; `sex` menjadi tepat dua nilai, **213 `Laki-laki`** (201 + 12 `'L'`) dan **96 `Perempuan`** (90 + 6 `' perempuan '`).
2. **`gabung_aman`:** tabel `klinik` lolos validasi many-to-one, jumlah baris tetap **309**, dan kunci yatim terlapor `['KL05']` (56 pasien dengan `kota` = NaN). `KL06` (Depok) tidak punya pasien sehingga tidak muncul.
3. **Agregasi kota × sex:** `groupby` membuang 56 baris `KL05` (kunci NaN) dan 17 baris lain dengan `chol` di luar 100–700, sehingga tabel hanya memakai **236 dari 309 baris**.
   * Rata-rata `chol` perempuan lebih tinggi di 3 dari 4 kota (Bandung 294,12 vs 256,48; Bekasi 269,85 vs 225,35; Bogor 240,55 vs 225,29) **tetapi terbalik di Jakarta** (239,94 vs 250,82), jadi polanya tidak konsisten.
   * Persentase `target = 1` pada perempuan lebih tinggi di keempat kota (63,6%–81,3%) dibanding laki-laki (40,5%–52,3%).
   * **Kehati-hatian:** ukuran sel hanya 16–44 pasien (satu pasien ≈ 2–6 poin persen), perbedaan tidak diuji secara statistik, dan ini bukan prevalensi populasi karena data berasal dari pasien rujukan.

---
### Soal B3 — Pembersih Otomatis yang Teruji *(20 poin | CPL-1 | CPMK-1 | Sub-CPMK-2 | 6 uji publik)*

In [7]:
def pilih_imputasi(df, kolom, kelompok='sex', proporsi=0.2, ulang=5, seed=42):
    """Memilih strategi imputasi terbaik ('median' atau 'median_kelompok') melalui eksperimen penyamaran.

    Sebagian nilai valid disamarkan (NaN), diimputasi dengan kedua metode, lalu dibandingkan
    dengan nilai sebenarnya memakai MAE rata-rata dari `ulang` pengulangan.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame masukan (tidak diubah).
    kolom : str
        Kolom numerik yang akan dievaluasi metode imputasinya.
    kelompok : str, default='sex'
        Kolom pengelompokan untuk median per kelompok.
    proporsi : float, default=0.2
        Proporsi nilai valid yang disamarkan pada tiap pengulangan.
    ulang : int, default=5
        Jumlah pengulangan eksperimen penyamaran.
    seed : int, default=42
        Seed generator bilangan acak.

    Returns
    -------
    str
        'median' atau 'median_kelompok' (MAE rata-rata terkecil; seri -> 'median').
    """
    if df is None or df.empty or kolom not in df.columns:
        raise ValueError(f"Kolom '{kolom}' tidak valid atau DataFrame kosong.")

    seri_num = pd.to_numeric(df[kolom], errors='coerce')
    idx_valid = seri_num.dropna().index.to_numpy()
    if len(idx_valid) < 2 or kelompok not in df.columns:
        return 'median'

    n_mask = max(1, int(round(len(idx_valid) * proporsi)))
    rng_imp = np.random.default_rng(seed)
    mae_global, mae_kelompok = [], []

    for _ in range(ulang):
        idx_samaran = rng_imp.choice(idx_valid, size=n_mask, replace=False)
        s_latih = seri_num.copy()
        nilai_sebenarnya = s_latih.loc[idx_samaran].to_numpy(dtype=float)
        s_latih.loc[idx_samaran] = np.nan

        med_glob = float(s_latih.median())
        mae_global.append(float(np.mean(np.abs(nilai_sebenarnya - med_glob))))

        med_per_grup = s_latih.groupby(df[kelompok]).median()
        pred_grup = df.loc[idx_samaran, kelompok].map(med_per_grup).fillna(med_glob).to_numpy(dtype=float)
        mae_kelompok.append(float(np.mean(np.abs(nilai_sebenarnya - pred_grup))))

    return 'median_kelompok' if np.mean(mae_kelompok) < np.mean(mae_global) else 'median'


def _kolom_numerik(df):
    """Daftar kolom numerik non-boolean pada df."""
    return [c for c in df.columns
            if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])]


def _tandai_hilang(df):
    """Langkah (a)-(d) pembersihan: seragamkan, buang duplikat, nol mustahil & nilai ekstrem -> NaN.

    Mengembalikan DataFrame BARU yang belum diimputasi (murni; df masukan tidak diubah).
    """
    df_kerja = seragamkan(df)                                                   # (a)
    kolom_non_id = [c for c in df_kerja.columns if c not in _kolom_pengenal(df_kerja)]
    df_kerja = df_kerja.drop_duplicates(subset=kolom_non_id, keep='first').reset_index(drop=True)  # (b)

    # (c) nol mustahil: kolomnya ditentukan oleh hasil deteksi_masalah, bukan ditulis langsung
    laporan = deteksi_masalah(df_kerja, None)
    for kol in laporan.loc[laporan['jenis'] == 'NOL_MUSTAHIL', 'kolom'].unique():
        df_kerja[kol] = df_kerja[kol].astype(float)
        df_kerja.loc[df_kerja[kol] == 0, kol] = np.nan

    # (d) nilai ekstrem (> Q3 + 10*IQR; aturan yang sama dengan deteksi_masalah) -> NaN
    for kol in _kolom_numerik(df_kerja):
        if df_kerja[kol].dropna().empty:
            continue
        _, _, iqr, batas = _pagar_ekstrem(df_kerja[kol])
        mask_ekstrem = df_kerja[kol] > batas
        if iqr > 0 and mask_ekstrem.any():
            df_kerja[kol] = df_kerja[kol].astype(float)
            df_kerja.loc[mask_ekstrem, kol] = np.nan
    return df_kerja


def bersihkan(df):
    """Pipeline pembersihan otomatis yang murni (df masukan tidak diubah) dan idempoten.

    Urutan: seragamkan -> buang duplikat (abaikan ID) -> nol mustahil & nilai ekstrem jadi NaN
    -> imputasi numerik dengan metode hasil pilih_imputasi -> assert validasi.

    Raises
    ------
    ValueError
        Bila ada kolom numerik yang seluruh nilainya kosong (tidak dapat diimputasi).
    """
    if df is None or df.empty:
        return df.copy() if df is not None else pd.DataFrame()

    df_bersih = _tandai_hilang(df)                                              # (a)-(d)
    kolom_num = _kolom_numerik(df_bersih)

    for kol in kolom_num:                                                       # (e)
        if df_bersih[kol].isna().all():
            raise ValueError(f"Kolom '{kol}' seluruhnya kosong; imputasi tidak mungkin dilakukan.")
        if not df_bersih[kol].isna().any():
            continue
        metode = pilih_imputasi(df_bersih, kol, kelompok='sex', proporsi=0.2, ulang=5, seed=42)
        med_glob = float(df_bersih[kol].median())
        if metode == 'median_kelompok' and 'sex' in df_bersih.columns:
            med_grup = df_bersih.groupby('sex')[kol].transform('median')
            df_bersih[kol] = df_bersih[kol].fillna(med_grup).fillna(med_glob)
        else:
            df_bersih[kol] = df_bersih[kol].fillna(med_glob)

    kolom_non_id = [c for c in df_bersih.columns if c not in _kolom_pengenal(df_bersih)]   # (f)
    assert df_bersih[kolom_num].isna().sum().sum() == 0, "Masih terdapat NaN pada kolom numerik!"
    assert df_bersih.duplicated(subset=kolom_non_id).sum() == 0, "Masih terdapat baris duplikat!"
    if 'sex' in df_bersih.columns:
        assert set(df_bersih['sex'].dropna().unique()).issubset({'Laki-laki', 'Perempuan'})
    return df_bersih


hd_bersih = bersihkan(hd)
hd_sebelum_imputasi = _tandai_hilang(hd)   # data setelah langkah (a)-(d), sebelum diimputasi

# Perbandingan deteksi outlier Z-score (|z| > 3) vs IQR (1.5 * IQR) pada chol & trestbps hasil pembersihan
perbandingan_outlier = []
for kol in ['chol', 'trestbps']:
    s = hd_bersih[kol]
    skor_z = (s - s.mean()) / s.std(ddof=1)
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    iqr = q3 - q1
    mask_iqr = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
    perbandingan_outlier.append({
        'kolom': kol,
        'NaN_diimputasi': int(hd_sebelum_imputasi[kol].isna().sum()),
        'metode_imputasi': pilih_imputasi(hd_sebelum_imputasi, kol),
        'mean': round(float(s.mean()), 2),
        'std': round(float(s.std(ddof=1)), 2),
        'median': round(float(s.median()), 2),
        'IQR': round(iqr, 2),
        'skewness': round(float(s.skew()), 2),
        'outlier_Z (|z|>3)': int((skor_z.abs() > 3.0).sum()),
        'outlier_IQR (1.5*IQR)': int(mask_iqr.sum()),
    })

df_outlier_b3 = pd.DataFrame(perbandingan_outlier)
display(df_outlier_b3)


,kolom,NaN_diimputasi,metode_imputasi,mean,std,median,IQR,skewness,outlier_Z (|z|>3),outlier_IQR (1.5*IQR)
0,chol,20,median_kelompok,246.49,50.87,240.0,61.5,1.22,4,5
1,trestbps,15,median,131.41,17.37,130.0,20.0,0.77,2,9


In [8]:
# Sel Uji Publik B3 (6 Uji Publik)
def _uji_b3():
    sekali = bersihkan(hd)
    dua_kali = bersihkan(sekali)
    uji_publik_b3 = [
        ("bersihkan bersifat murni (tidak mengubah hd)",
         lambda: uji(hd.equals(HD_ASLI))),
        ("bersihkan bersifat idempoten: bersihkan(bersihkan(hd)) == bersihkan(hd)",
         lambda: uji(sekali.equals(dua_kali))),
        ("Duplikat non-ID terhapus seluruhnya (tersisa 303 baris unik)",
         lambda: uji(len(sekali) == 303 and sekali.duplicated(subset=[c for c in sekali.columns if c != 'id_pasien']).sum() == 0)),
        ("Tidak ada lagi nilai NaN di seluruh kolom",
         lambda: uji(int(sekali.isna().sum().sum()) == 0)),
        ("Nol mustahil (0) dan galat input (5640) pada 'chol' sudah terimputasi (rentang 100-600)",
         lambda: uji(sekali['chol'].between(100, 600).all())),
        ("pilih_imputasi mengembalikan opsi sah ('median' atau 'median_kelompok')",
         lambda: uji(pilih_imputasi(sekali, 'chol') in {'median', 'median_kelompok'}
                     and pilih_imputasi(sekali, 'trestbps') in {'median', 'median_kelompok'})),
    ]
    jalankan_uji("B3 - Pembersih Otomatis yang Teruji", uji_publik_b3)

_uji_b3()

=== Menjalankan Uji Publik B3 - Pembersih Otomatis yang Teruji (6 uji) ===
  [LULUS 1/6] bersihkan bersifat murni (tidak mengubah hd)
  [LULUS 2/6] bersihkan bersifat idempoten: bersihkan(bersihkan(hd)) == bersihkan(hd)
  [LULUS 3/6] Duplikat non-ID terhapus seluruhnya (tersisa 303 baris unik)
  [LULUS 4/6] Tidak ada lagi nilai NaN di seluruh kolom
  [LULUS 5/6] Nol mustahil (0) dan galat input (5640) pada 'chol' sudah terimputasi (rentang 100-600)
  [LULUS 6/6] pilih_imputasi mengembalikan opsi sah ('median' atau 'median_kelompok')
Hasil B3 - Pembersih Otomatis yang Teruji: 6/6 uji publik LULUS.



In [9]:
# Uji tambahan B3 (kasus tepi buatan sendiri; di luar 6 uji publik soal)
def _uji_b3_tepi():
    def _kolom_kosong_ditolak():
        try:
            bersihkan(hd.assign(trestbps=np.nan))
            return False
        except ValueError:
            return True

    kecil = hd.head(10)
    uji_tepi = [
        ("Tabel kecil (10 baris): murni dan idempoten",
         lambda: uji(bersihkan(kecil).equals(bersihkan(bersihkan(kecil))) and kecil.equals(HD_ASLI.head(10)))),
        ("Kolom 'sex' dan 'chol' tidak ada: tetap berjalan tanpa NaN tersisa",
         lambda: uji(not bersihkan(hd.drop(columns=['sex', 'chol'])).isna().any().any())),
        ("Kolom numerik yang seluruhnya kosong ditolak dengan ValueError",
         lambda: uji(_kolom_kosong_ditolak())),
    ]
    jalankan_uji("B3 - uji tambahan kasus tepi", uji_tepi)

_uji_b3_tepi()

=== Menjalankan Uji Publik B3 - uji tambahan kasus tepi (3 uji) ===
  [LULUS 1/3] Tabel kecil (10 baris): murni dan idempoten


  [LULUS 2/3] Kolom 'sex' dan 'chol' tidak ada: tetap berjalan tanpa NaN tersisa
  [LULUS 3/3] Kolom numerik yang seluruhnya kosong ditolak dengan ValueError
Hasil B3 - uji tambahan kasus tepi: 3/3 uji publik LULUS.



#### Interpretasi B3:
1. **Hasil pipeline:** 309 → **303 baris** (6 duplikat dibuang). Yang diimputasi: `chol` **20** nilai (18 nol mustahil + 2 galat `5640`) dan `trestbps` **15** nilai. Angkanya 15, bukan 16, karena satu NaN berada pada baris duplikat yang sudah dibuang lebih dulu. Uji publik membuktikan fungsi murni dan idempoten (`bersihkan(bersihkan(hd))` identik).
2. **Metode imputasi hasil eksperimen penyamaran:** `chol` → `median_kelompok`, `trestbps` → `median`. Namun MAE-nya nyaris sama: `chol` 42,01 (per sex) vs 42,15 (global), selisih 0,14 mg/dL (< 0,4%); `trestbps` 13,57 vs 13,42. Median per sex hampir identik (`chol` 239 vs 244; `trestbps` 130 vs 130), jadi pilihan ini berada di batas derau dan tidak berdampak praktis. Efek samping yang nyata: 20 nilai `chol` hanya diisi dua angka (239 muncul 17 kali, 244 muncul 10 kali), sehingga menimbulkan lonjakan di tengah distribusi dan sedikit menyusutkan varians.
3. **Z-score vs IQR (data hasil pembersihan):**

   | Kolom | Std | IQR | Skewness | Outlier Z (\|z\|>3) | Outlier IQR (1,5·IQR) |
   |---|---|---|---|---|---|
   | `chol` | 50,87 | 61,5 | 1,22 | 4 | 5 |
   | `trestbps` | 17,37 | 20,0 | 0,77 | 2 | 9 |

   IQR menandai lebih banyak outlier, terutama `trestbps` (9 vs 2). Penyebabnya: mean dan simpangan baku ikut terseret ekor kanan (std 17,37 lebih besar daripada IQR/1,349 ≈ 14,8) sehingga ambang $|z|>3$ menjadi longgar, sedangkan kuartil tahan terhadap ekor. Karena kedua kolom menceng ke kanan, pagar IQR sisi atas cenderung menandai banyak titik. Nilai tinggi itu bisa saja kondisi klinis sah (hipertensi, hiperkolesterolemia), jadi **ditandai untuk ditinjau, bukan otomatis dibuang**. Perbandingan ini dihitung setelah imputasi, sehingga lonjakan nilai median ikut memengaruhi sebaran.

---
### Soal B4 — Transformer Sendiri dengan Kontrak `fit/transform` *(10 poin | CPL-2 | CPMK-1 | Sub-CPMK-2 | 6 uji publik)*

In [10]:
from sklearn.model_selection import train_test_split


class SkalaRobustKu:
    """Transformer penskalaan robust ( (x - median) / IQR ) dengan kontrak fit/transform."""

    def __init__(self, kolom):
        self.kolom = list(kolom)

    def fit(self, X, y=None):
        """Mempelajari median_ dan iqr_ untuk setiap kolom pada self.kolom."""
        if not isinstance(X, pd.DataFrame):
            raise ValueError("Input X harus berupa pandas DataFrame.")
        self.median_ = {}
        self.iqr_ = {}
        for kol in self.kolom:
            if kol not in X.columns:
                raise ValueError(f"Kolom '{kol}' tidak ditemukan pada DataFrame X.")
            s = pd.to_numeric(X[kol], errors='coerce').dropna()
            if s.empty:
                self.median_[kol] = 0.0
                self.iqr_[kol] = 1.0
            else:
                med = float(s.median())
                iqr = float(s.quantile(0.75) - s.quantile(0.25))
                self.median_[kol] = med
                # Hindari pembagian dengan nol bila IQR == 0
                self.iqr_[kol] = iqr if iqr != 0.0 else 1.0
        return self

    def transform(self, X):
        """Menerapkan transformasi robust scaling menggunakan median_ dan iqr_ hasil fit."""
        if not hasattr(self, 'median_') or not hasattr(self, 'iqr_'):
            raise RuntimeError("SkalaRobustKu belum di-fit. Panggil fit(X) terlebih dahulu.")
        if not isinstance(X, pd.DataFrame):
            raise ValueError("Input X harus berupa pandas DataFrame.")

        hasil = pd.DataFrame(index=X.index)
        for kol in self.kolom:
            if kol not in X.columns:
                raise ValueError(f"Kolom '{kol}' tidak ditemukan pada DataFrame X.")
            hasil[kol] = (
                pd.to_numeric(X[kol], errors='coerce') - self.median_[kol]
            ) / self.iqr_[kol]
        return hasil


class OneHotKu:
    """Transformer One-Hot Encoding yang menjamin konsistensi kolom antara data latih dan uji."""

    def __init__(self, kolom):
        self.kolom = list(kolom)

    def fit(self, X, y=None):
        """Mempelajari daftar kategori_ unik terurut dan nama kolom_fitur_ dari data latih."""
        if not isinstance(X, pd.DataFrame):
            raise ValueError("Input X harus berupa pandas DataFrame.")
        self.kategori_ = {}
        self.kolom_fitur_ = []
        for kol in self.kolom:
            if kol not in X.columns:
                raise ValueError(f"Kolom '{kol}' tidak ditemukan pada DataFrame X.")
            kat_urut = sorted(X[kol].dropna().unique(), key=lambda v: str(v))
            self.kategori_[kol] = kat_urut
            for kat in kat_urut:
                self.kolom_fitur_.append(f"{kol}_{kat}")
        return self

    def transform(self, X):
        """Menghasilkan DataFrame biner one-hot dengan skema kolom identik hasil fit."""
        if not hasattr(self, 'kategori_') or not hasattr(self, 'kolom_fitur_'):
            raise RuntimeError("OneHotKu belum di-fit. Panggil fit(X) terlebih dahulu.")
        if not isinstance(X, pd.DataFrame):
            raise ValueError("Input X harus berupa pandas DataFrame.")

        hasil = pd.DataFrame(0, index=X.index, columns=self.kolom_fitur_, dtype=int)
        for kol in self.kolom:
            if kol not in X.columns:
                raise ValueError(f"Kolom '{kol}' tidak ditemukan pada DataFrame X.")
            for kat in self.kategori_[kol]:
                hasil[f"{kol}_{kat}"] = (X[kol] == kat).astype(int)
        return hasil


# 1. Pembagian data bersih 75/25 dengan stratify pada 'target' dan random_state=42
df_latih, df_uji = train_test_split(
    hd_bersih,
    test_size=0.25,
    stratify=hd_bersih['target'],
    random_state=42,
)

# 5. Bangun matriks fitur latih dan uji
KOLOM_NUMERIK_B4 = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
KOLOM_KATEGORIKAL_B4 = ['cp', 'thal', 'sex']

skala_b4 = SkalaRobustKu(KOLOM_NUMERIK_B4).fit(df_latih)
onehot_b4 = OneHotKu(KOLOM_KATEGORIKAL_B4).fit(df_latih)

X_latih = pd.concat([skala_b4.transform(df_latih), onehot_b4.transform(df_latih)], axis=1)
X_uji = pd.concat([skala_b4.transform(df_uji), onehot_b4.transform(df_uji)], axis=1)
y_latih = df_latih['target'].to_numpy(dtype=int)
y_uji = df_uji['target'].to_numpy(dtype=int)

print(f"X_latih: {X_latih.shape} | X_uji: {X_uji.shape}")
display(X_latih.head())

X_latih: (227, 15) | X_uji: (76, 15)


,age,trestbps,chol,thalach,oldpeak,cp_0,cp_1,cp_2,cp_3,thal_0,thal_1,thal_2,thal_3,sex_Laki-laki,sex_Perempuan
66,-0.285714,-1.5,-0.321429,-0.305085,0.222222,0,0,1,0,0,0,1,0,1,0
260,0.785714,2.4,0.071429,0.440678,0.111111,1,0,0,0,0,0,0,1,0,1
289,0.000000,-0.1,-0.625000,-0.745763,0.666667,1,0,0,0,0,0,0,1,0,1
237,0.357143,0.5,0.946429,0.610169,0.222222,1,0,0,0,0,0,0,1,1,0
144,1.500000,0.5,-0.767857,-1.220339,0.166667,0,0,1,0,0,0,1,0,0,1


In [11]:
# Sel Uji Publik B4 (6 Uji Publik)
def _uji_b4():
    salinan_latih = df_latih.copy()
    _ = skala_b4.transform(df_latih)

    def _cek_runtime_error_sebelum_fit():
        try:
            SkalaRobustKu(['age']).transform(df_latih)
            return False
        except RuntimeError:
            pass
        try:
            OneHotKu(['sex']).transform(df_latih)
            return False
        except RuntimeError:
            return True

    # Simulasi kategori baru pada data uji
    df_uji_asing = df_uji.copy()
    df_uji_asing.iloc[0, df_uji_asing.columns.get_loc('sex')] = 'Kategori_Baru_Asing'
    oh_asing = onehot_b4.transform(df_uji_asing)

    uji_publik_b4 = [
        ("Proporsi split 75/25 terstratifikasi (227 latih, 76 uji)",
         lambda: uji(len(df_latih) == 227 and len(df_uji) == 76)),
        ("Melempar RuntimeError bila transform dipanggil sebelum fit",
         lambda: uji(_cek_runtime_error_sebelum_fit())),
        ("Memiliki atribut hasil belajar berakhiran '_' (median_, iqr_, kategori_)",
         lambda: uji(hasattr(skala_b4, 'median_') and hasattr(skala_b4, 'iqr_') and hasattr(onehot_b4, 'kategori_'))),
        ("Median fitur numerik pada X_latih hasil SkalaRobustKu bernilai ~0.0",
         lambda: uji(np.allclose(X_latih[KOLOM_NUMERIK_B4].median().to_numpy(), 0.0, atol=1e-12))),
        ("OneHotKu menghasilkan kolom identik pada latih, uji, dan saat ada kategori baru",
         lambda: uji(list(X_latih.columns) == list(X_uji.columns)
                     and list(oh_asing.columns) == onehot_b4.kolom_fitur_
                     and int(oh_asing.iloc[0][['sex_Laki-laki', 'sex_Perempuan']].sum()) == 0)),
        ("Tidak mengubah DataFrame masukan df_latih (fungsi/metode murni)",
         lambda: uji(df_latih.equals(salinan_latih))),
    ]
    jalankan_uji("B4 - Transformer Kontrak fit/transform", uji_publik_b4)

_uji_b4()

=== Menjalankan Uji Publik B4 - Transformer Kontrak fit/transform (6 uji) ===
  [LULUS 1/6] Proporsi split 75/25 terstratifikasi (227 latih, 76 uji)
  [LULUS 2/6] Melempar RuntimeError bila transform dipanggil sebelum fit
  [LULUS 3/6] Memiliki atribut hasil belajar berakhiran '_' (median_, iqr_, kategori_)
  [LULUS 4/6] Median fitur numerik pada X_latih hasil SkalaRobustKu bernilai ~0.0
  [LULUS 5/6] OneHotKu menghasilkan kolom identik pada latih, uji, dan saat ada kategori baru
  [LULUS 6/6] Tidak mengubah DataFrame masukan df_latih (fungsi/metode murni)
Hasil B4 - Transformer Kontrak fit/transform: 6/6 uji publik LULUS.



#### Interpretasi B4:
1. **Pembagian & pencegahan kebocoran:** 303 baris menjadi **227 latih / 76 uji**. Proporsi `target = 1` terjaga: 54,6% (124/227) pada latih dan 53,9% (41/76) pada uji, bukti stratifikasi bekerja. `median_`, `iqr_`, dan `kategori_` dipelajari hanya dari `df_latih`.
2. **Matriks fitur:** `X_latih` berukuran (227, 15) dan `X_uji` (76, 15) dengan kolom identik: 5 fitur numerik terskala + 10 indikator (`cp` 4, `thal` 4, `sex` 2). Median tiap fitur numerik pada latih = 0, sesuai definisi skala robust. Kategori yang tak pernah terlihat saat `fit` menghasilkan baris nol, bukan galat dan bukan kolom baru.
3. **Keterbatasan yang perlu diakui:** imputasi median di `bersihkan` dihitung dari seluruh 303 baris *sebelum* split, sehingga sedikit informasi data uji ikut masuk ke nilai imputasi (kebocoran ringan). Soal memang meminta pembersihan sebelum split; perbaikan idealnya imputasi dijadikan transformer ketiga yang di-`fit` pada data latih saja. Kolom `sex_Laki-laki` dan `sex_Perempuan` saling berkolinear sempurna, tidak masalah untuk regresi logistik teregularisasi (B6), tetapi untuk OLS (B5) dipakai `drop_first`.

---
### Soal B5 — Regresi Linear dari Nol *(10 poin | CPL-1 | CPMK-2 | Sub-CPMK-3 | 4 uji publik)*

In [12]:
def ols(X, y):
    """Mengestimasi parameter Ordinary Least Squares (intersep, koef) memakai np.linalg.lstsq."""
    X_arr = np.asarray(X, dtype=float)
    y_arr = np.asarray(y, dtype=float).ravel()
    if X_arr.ndim == 1:
        X_arr = X_arr.reshape(-1, 1)
    if len(X_arr) != len(y_arr) or len(y_arr) == 0:
        raise ValueError("Panjang X dan y harus sama dan tidak boleh kosong.")

    # Tambahkan kolom 1 di sebelah kiri untuk intersep
    X_desain = np.column_stack([np.ones(len(X_arr), dtype=float), X_arr])
    theta, _, _, _ = np.linalg.lstsq(X_desain, y_arr, rcond=None)
    intersep = float(theta[0])
    koef = np.asarray(theta[1:], dtype=float)
    return intersep, koef


def metrik_regresi(y, yhat):
    """Menghitung metrik evaluasi regresi {'R²', 'MAE', 'RMSE'} dari nol."""
    y_arr = np.asarray(y, dtype=float).ravel()
    yhat_arr = np.asarray(yhat, dtype=float).ravel()
    if len(y_arr) != len(yhat_arr) or len(y_arr) == 0:
        raise ValueError("Panjang y dan yhat harus sama dan tidak boleh kosong.")

    residual = y_arr - yhat_arr
    ss_res = float(np.sum(residual ** 2))
    ss_tot = float(np.sum((y_arr - np.mean(y_arr)) ** 2))
    r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0.0 else 0.0
    mae = float(np.mean(np.abs(residual)))
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    return {'R²': r2, 'MAE': mae, 'RMSE': rmse}


# 3. Prediksi 'thalach' dari 'age', 'trestbps', 'chol', 'exang', dan 'sex' (dummy, drop_first)
#    pada split B4 menggunakan data sebelum scaling
def _bangun_fitur_b5(df_bagian):
    fitur_num = df_bagian[['age', 'trestbps', 'chol', 'exang']].copy().astype(float)
    dummy_sex = pd.get_dummies(df_bagian['sex'], prefix='sex', drop_first=True, dtype=float)
    return pd.concat([fitur_num, dummy_sex], axis=1)


X_latih_b5 = _bangun_fitur_b5(df_latih)
X_uji_b5 = _bangun_fitur_b5(df_uji).reindex(columns=X_latih_b5.columns, fill_value=0.0)
y_latih_b5 = df_latih['thalach'].to_numpy(dtype=float)
y_uji_b5 = df_uji['thalach'].to_numpy(dtype=float)

intersep_ols, koef_ols = ols(X_latih_b5, y_latih_b5)
yhat_uji_ols = intersep_ols + X_uji_b5.to_numpy(dtype=float) @ koef_ols

# 4. Tolok ukur (baseline) rata-rata y_latih
yhat_uji_baseline = np.full_like(y_uji_b5, fill_value=np.mean(y_latih_b5))

tabel_koef_b5 = pd.DataFrame({
    'Fitur': ['(Intersep)'] + list(X_latih_b5.columns),
    'Estimasi_Parameter': [intersep_ols] + list(koef_ols)
}).round(4)
display(tabel_koef_b5)

tabel_perbandingan_b5 = pd.DataFrame([
    {'Model': 'Regresi Linear OLS', **metrik_regresi(y_uji_b5, yhat_uji_ols)},
    {'Model': 'Tolok Ukur Rata-rata (Baseline)', **metrik_regresi(y_uji_b5, yhat_uji_baseline)},
]).round(4)
display(tabel_perbandingan_b5)

,Fitur,Estimasi_Parameter
0,(Intersep),191.3786
1,age,-0.9830
2,trestbps,0.0918
3,chol,0.0198
4,exang,-15.2841
5,sex_Perempuan,-0.3906


,Model,R²,MAE,RMSE
0,Regresi Linear OLS,0.2983,15.7229,20.7413
1,Tolok Ukur Rata-rata (Baseline),-0.0004,19.9012,24.7654


In [13]:
# Sel Uji Publik B5 (4 Uji Publik)
def _uji_b5():
    # Uji kecocokan eksak pada relasi linier sintetis y = 5 + 2*x1 - 3*x2
    rng_tes = np.random.default_rng(42)
    X_sin = rng_tes.normal(size=(50, 2))
    y_sin = 5.0 + 2.0 * X_sin[:, 0] - 3.0 * X_sin[:, 1]
    b0_sin, b_sin = ols(X_sin, y_sin)

    def _cek_error_panjang_berbeda():
        try:
            metrik_regresi([1.0, 2.0], [1.0])
            return False
        except ValueError:
            return True

    m_ols = metrik_regresi(y_uji_b5, yhat_uji_ols)
    m_base = metrik_regresi(y_uji_b5, yhat_uji_baseline)

    uji_publik_b5 = [
        ("ols memulihkan intersep dan koefisien data sintetis dengan tepat",
         lambda: uji(np.isclose(b0_sin, 5.0) and np.allclose(b_sin, [2.0, -3.0]))),
        ("metrik_regresi mengembalikan dict {'R²', 'MAE', 'RMSE'} dengan RMSE >= MAE",
         lambda: uji(set(m_ols.keys()) == {'R²', 'MAE', 'RMSE'} and m_ols['RMSE'] >= m_ols['MAE'])),
        ("metrik_regresi melempar ValueError bila panjang y dan yhat berbeda",
         lambda: uji(_cek_error_panjang_berbeda())),
        ("Model OLS mengungguli tolok ukur rata-rata pada data uji (R2 lebih tinggi & RMSE lebih rendah)",
         lambda: uji(m_ols['R²'] > m_base['R²'] and m_ols['RMSE'] < m_base['RMSE'])),
    ]
    jalankan_uji("B5 - Regresi Linear dari Nol", uji_publik_b5)

_uji_b5()

=== Menjalankan Uji Publik B5 - Regresi Linear dari Nol (4 uji) ===
  [LULUS 1/4] ols memulihkan intersep dan koefisien data sintetis dengan tepat
  [LULUS 2/4] metrik_regresi mengembalikan dict {'R²', 'MAE', 'RMSE'} dengan RMSE >= MAE
  [LULUS 3/4] metrik_regresi melempar ValueError bila panjang y dan yhat berbeda
  [LULUS 4/4] Model OLS mengungguli tolok ukur rata-rata pada data uji (R2 lebih tinggi & RMSE lebih rendah)
Hasil B5 - Regresi Linear dari Nol: 4/4 uji publik LULUS.



#### Interpretasi B5:
1. **Dibanding tolok ukur rata-rata (data uji, n = 76):**

   | Model | R² | MAE | RMSE |
   |---|---|---|---|
   | OLS | 0,2983 | 15,72 | 20,74 |
   | Rata-rata `y_latih` | −0,0004 | 19,90 | 24,77 |

   OLS menurunkan MAE sekitar 21% dan RMSE sekitar 16%, dan menjelaskan ±30% variasi `thalach` pada data yang tidak dipakai melatih. Artinya ada sinyal nyata, tetapi **70% variasi tetap tak terjelaskan** (variabel seperti `slope`, `oldpeak`, atau obat tidak dimasukkan). Dengan 76 data uji, angka R² ini masih cukup berisik.
2. **Koefisien `age` = −0,983 bpm per tahun** (ceteris paribus terhadap `trestbps`, `chol`, `exang`, `sex`): tiap tambahan usia setahun berasosiasi dengan penurunan denyut jantung maksimum sekitar 1 detak/menit, mendekati aturan klasik $220-\text{usia}$ (kemiringan −1). Ini asosiasi lintas-pasien, bukan efek kausal individu, dan tanpa selang kepercayaan/uji signifikansi kita tidak boleh menyebutnya "signifikan".
3. **Koefisien lain:** `exang` = −15,28 bpm (terbesar; pasien dengan angina akibat olahraga mencapai denyut maksimum lebih rendah), sedangkan `trestbps` (+0,09), `chol` (+0,02), dan `sex_Perempuan` (−0,39) praktis mendekati nol. Intersep 191,38 bpm adalah ekstrapolasi ke usia 0 dan tidak bermakna sendiri.

---
### Soal B6 — Regresi Logistik, AUC, dan Pemilihan Ambang *(15 poin | CPL-2 | CPMK-2 | Sub-CPMK-3, Sub-CPMK-4 | 5 uji publik)*

In [14]:
from sklearn.linear_model import LogisticRegression


def auc_peringkat(y, skor):
    """Menghitung ROC-AUC berbasis statistik Mann-Whitney U dengan penanganan skor kembar (ties)."""
    y_arr = np.asarray(y, dtype=int).ravel()
    skor_arr = np.asarray(skor, dtype=float).ravel()
    if len(y_arr) != len(skor_arr) or len(y_arr) == 0:
        raise ValueError("Panjang y dan skor harus sama dan tidak boleh kosong.")

    mask_pos = (y_arr == 1)
    mask_neg = (y_arr == 0)
    n_pos = int(np.sum(mask_pos))
    n_neg = int(np.sum(mask_neg))
    if n_pos == 0 or n_neg == 0:
        raise ValueError("y harus memuat minimal satu sampel positif (1) dan satu negatif (0).")

    # Urutkan skor dan berikan peringkat rata-rata (1-based) untuk nilai yang kembar
    n = len(skor_arr)
    urutan = np.argsort(skor_arr, kind='mergesort')
    skor_urut = skor_arr[urutan]
    peringkat_urut = np.empty(n, dtype=float)

    i = 0
    while i < n:
        j = i
        while j + 1 < n and skor_urut[j + 1] == skor_urut[i]:
            j += 1
        peringkat_urut[i:j + 1] = 0.5 * ((i + 1) + (j + 1))
        i = j + 1

    peringkat = np.empty(n, dtype=float)
    peringkat[urutan] = peringkat_urut

    r_pos = float(np.sum(peringkat[mask_pos]))
    u_pos = r_pos - (n_pos * (n_pos + 1) / 2.0)
    return float(u_pos / (n_pos * n_neg))


def lipatan_berstrata(y, k=5, seed=42):
    """Generator lipatan validasi silang berstrata (menghasilkan tuple (indeks_latih, indeks_val))."""
    y_arr = np.asarray(y).ravel()
    if k < 2 or k > len(y_arr):
        raise ValueError("Jumlah lipatan k harus memenuhi 2 <= k <= len(y).")

    rng_cv = np.random.default_rng(seed)
    kantong_val = [[] for _ in range(k)]

    for kelas in np.unique(y_arr):
        idx_kelas = np.flatnonzero(y_arr == kelas).copy()
        rng_cv.shuffle(idx_kelas)
        for i, bagian in enumerate(np.array_split(idx_kelas, k)):
            kantong_val[i].append(bagian)

    semua_indeks = np.arange(len(y_arr))
    for i in range(k):
        idx_val = np.sort(np.concatenate(kantong_val[i]))
        mask_val = np.zeros(len(y_arr), dtype=bool)
        mask_val[idx_val] = True
        idx_latih = semua_indeks[~mask_val]
        yield idx_latih, idx_val


def ambang_untuk_recall(y, prob, target):
    """Memilih ambang probabilitas tertinggi yang mencapai recall >= target."""
    y_arr = np.asarray(y, dtype=int).ravel()
    prob_arr = np.asarray(prob, dtype=float).ravel()
    if len(y_arr) != len(prob_arr) or len(y_arr) == 0:
        raise ValueError("Panjang y dan prob harus sama dan tidak boleh kosong.")
    if not (0.0 <= target <= 1.0):
        raise ValueError("Target recall harus berada pada rentang [0, 1].")

    n_pos = int(np.sum(y_arr == 1))
    if n_pos == 0:
        raise ValueError("Tidak terdapat sampel kelas positif pada y.")

    kandidat = np.sort(np.unique(np.concatenate([[0.0], prob_arr])))[::-1]
    for t in kandidat:
        pred = (prob_arr >= t).astype(int)
        recall = float(np.sum((y_arr == 1) & (pred == 1))) / n_pos
        if recall >= target:
            return float(t)
    return 0.0


# (b) Pemilihan C terbaik dengan 5-fold Stratified CV pada matriks fitur latih
KANDIDAT_C = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
X_latih_mat = X_latih.to_numpy(dtype=float)
X_uji_mat = X_uji.to_numpy(dtype=float)

rekap_cv = []
oof_prob_per_c = {}

for nilai_c in KANDIDAT_C:
    auc_per_lipatan = []
    prob_oof = np.zeros(len(y_latih), dtype=float)
    for idx_tr, idx_va in lipatan_berstrata(y_latih, k=5, seed=42):
        model_cv = LogisticRegression(C=nilai_c, max_iter=1000, random_state=42)
        model_cv.fit(X_latih_mat[idx_tr], y_latih[idx_tr])
        prob_va = model_cv.predict_proba(X_latih_mat[idx_va])[:, 1]
        prob_oof[idx_va] = prob_va
        auc_per_lipatan.append(auc_peringkat(y_latih[idx_va], prob_va))

    oof_prob_per_c[nilai_c] = prob_oof
    rekap_cv.append({
        'C': nilai_c,
        'Mean_CV_AUC': float(np.mean(auc_per_lipatan)),
        'Std_CV_AUC': float(np.std(auc_per_lipatan, ddof=1)),
        'OOF_AUC': auc_peringkat(y_latih, prob_oof),
    })

df_rekap_cv = pd.DataFrame(rekap_cv)
display(df_rekap_cv.round(4))

baris_terbaik = df_rekap_cv.sort_values('Mean_CV_AUC', ascending=False).iloc[0]
C_TERBAIK = float(baris_terbaik['C'])

# (c) Tentukan ambang pada prediksi out-of-fold (OOF) data latih untuk target recall >= 0.90
prob_oof_terbaik = oof_prob_per_c[C_TERBAIK]
AMBANG_TERPILIH = ambang_untuk_recall(y_latih, prob_oof_terbaik, target=0.90)
print(f"C Terbaik: {C_TERBAIK} | Ambang OOF (Recall >= 0.90): {AMBANG_TERPILIH:.4f}")

# (d) Evaluasi SATU KALI SAJA pada data uji
model_logistik_final = LogisticRegression(C=C_TERBAIK, max_iter=1000, random_state=42)
model_logistik_final.fit(X_latih_mat, y_latih)

prob_uji_final = model_logistik_final.predict_proba(X_uji_mat)[:, 1]
pred_uji_final = (prob_uji_final >= AMBANG_TERPILIH).astype(int)

tn_u = int(np.sum((y_uji == 0) & (pred_uji_final == 0)))
fp_u = int(np.sum((y_uji == 0) & (pred_uji_final == 1)))
fn_u = int(np.sum((y_uji == 1) & (pred_uji_final == 0)))
tp_u = int(np.sum((y_uji == 1) & (pred_uji_final == 1)))

presisi_u = tp_u / (tp_u + fp_u) if (tp_u + fp_u) > 0 else 0.0
recall_u = tp_u / (tp_u + fn_u) if (tp_u + fn_u) > 0 else 0.0
f1_u = (2.0 * presisi_u * recall_u / (presisi_u + recall_u)) if (presisi_u + recall_u) > 0 else 0.0
auc_u = auc_peringkat(y_uji, prob_uji_final)

df_cm_uji = pd.DataFrame(
    [[tn_u, fp_u], [fn_u, tp_u]],
    index=['Aktual 0 (Sehat)', 'Aktual 1 (Sakit)'],
    columns=['Prediksi 0 (Sehat)', 'Prediksi 1 (Sakit)']
)
display(df_cm_uji)

df_metrik_uji_b6 = pd.DataFrame([{
    'C_Terbaik': C_TERBAIK,
    'Ambang_OOF': round(AMBANG_TERPILIH, 4),
    'Presisi_Uji': round(presisi_u, 4),
    'Recall_Uji': round(recall_u, 4),
    'F1_Uji': round(f1_u, 4),
    'AUC_Uji': round(auc_u, 4),
}])
display(df_metrik_uji_b6)

,C,Mean_CV_AUC,Std_CV_AUC,OOF_AUC
0,0.01,0.9061,0.0557,0.9052
1,0.03,0.9019,0.0483,0.9049
2,0.10,0.9028,0.0467,0.9055
3,0.30,0.9018,0.0412,0.9035
4,1.00,0.8996,0.0443,0.8999
5,3.00,0.8998,0.0424,0.8979
6,10.00,0.8973,0.0439,0.8964


C Terbaik: 0.01 | Ambang OOF (Recall >= 0.90): 0.5027


,Prediksi 0 (Sehat),Prediksi 1 (Sakit)
Aktual 0 (Sehat),20,15
Aktual 1 (Sakit),4,37


,C_Terbaik,Ambang_OOF,Presisi_Uji,Recall_Uji,F1_Uji,AUC_Uji
0,0.01,0.5027,0.7115,0.9024,0.7957,0.8474


In [15]:
# Sel Uji Publik B6 (5 Uji Publik)
def _uji_b6():
    # 1. Uji auc_peringkat pada kasus normal & skor kembar (ties)
    y_tie = np.array([0, 0, 1, 1])
    skor_tie = np.array([0.1, 0.5, 0.5, 0.9])
    # Pasangan: (1,0): (0.5 vs 0.1 -> 1), (0.5 vs 0.5 -> 0.5), (0.9 vs 0.1 -> 1), (0.9 vs 0.5 -> 1) => 3.5 / 4 = 0.875

    # 2. Uji generator lipatan_berstrata
    lipatan = list(lipatan_berstrata(y_latih, k=5, seed=42))
    semua_val = np.concatenate([ idx_v for _, idx_v in lipatan ])

    # 3. Uji ambang_untuk_recall pada OOF
    recall_oof = float(np.sum((y_latih == 1) & (prob_oof_terbaik >= AMBANG_TERPILIH))) / int(np.sum(y_latih == 1))

    uji_publik_b6 = [
        ("auc_peringkat menangani skor sempurna (1.0) dan skor kembar (0.875) dengan tepat",
         lambda: uji(np.isclose(auc_peringkat([0, 1, 0, 1], [0.1, 0.8, 0.2, 0.9]), 1.0)
                     and np.isclose(auc_peringkat(y_tie, skor_tie), 0.875))),
        ("lipatan_berstrata merupakan generator 5 lipatan yang saling lepas dan mencakup seluruh indeks",
         lambda: uji(len(lipatan) == 5 and sorted(semua_val.tolist()) == list(range(len(y_latih))))),
        ("Setiap lipatan berstrata menjaga proporsi kelas positif mendekati proporsi populasi",
         lambda: uji(all( abs(np.mean(y_latih[idx_v]) - np.mean(y_latih)) < 0.05 for _, idx_v in lipatan ))),
        ("ambang_untuk_recall mencapai recall >= 0.90 pada prediksi OOF data latih",
         lambda: uji(recall_oof >= 0.90 and 0.0 < AMBANG_TERPILIH <= 1.0)),
        ("Evaluasi final pada data uji menghasilkan AUC > 0.80 dan Recall tinggi",
         lambda: uji(auc_u > 0.80 and recall_u >= 0.80)),
    ]
    jalankan_uji("B6 - Regresi Logistik, AUC, dan Pemilihan Ambang", uji_publik_b6)

_uji_b6()

=== Menjalankan Uji Publik B6 - Regresi Logistik, AUC, dan Pemilihan Ambang (5 uji) ===
  [LULUS 1/5] auc_peringkat menangani skor sempurna (1.0) dan skor kembar (0.875) dengan tepat
  [LULUS 2/5] lipatan_berstrata merupakan generator 5 lipatan yang saling lepas dan mencakup seluruh indeks
  [LULUS 3/5] Setiap lipatan berstrata menjaga proporsi kelas positif mendekati proporsi populasi
  [LULUS 4/5] ambang_untuk_recall mencapai recall >= 0.90 pada prediksi OOF data latih
  [LULUS 5/5] Evaluasi final pada data uji menghasilkan AUC > 0.80 dan Recall tinggi
Hasil B6 - Regresi Logistik, AUC, dan Pemilihan Ambang: 5/5 uji publik LULUS.



#### Interpretasi B6:
1. **Pemilihan $C$ (CV 5-lipatan buatan sendiri, hanya data latih):** AUC rata-rata berkisar 0,8973–0,9061; terbaik **$C = 0{,}01$** (0,9061 ± 0,0557). Selisih antar-$C$ (≤ 0,009) jauh lebih kecil daripada simpangan baku antar-lipatan (0,041–0,056), jadi semua $C$ pada dasarnya setara; $C = 0{,}01$ menang tipis dan berada di **ujung grid** (regularisasi terkuat), sehingga bila ingin yakin, grid perlu diperluas ke $C < 0{,}01$.
2. **Ambang:** dari prediksi out-of-fold, ambang tertinggi dengan recall ≥ 0,90 adalah **0,5027**, nyaris sama dengan 0,5 karena regularisasi kuat memampatkan probabilitas ke sekitar 0,5.
3. **Data uji (sekali pakai, n = 76):** TN = 20, FP = 15, FN = 4, TP = 37 → **presisi 0,7115, recall 0,9024, F1 0,7957, AUC 0,8474**; spesifisitas = 20/35 = 0,571. Recall uji (0,902) hampir persis target 0,90, jadi ambang dari data latih terbawa baik ke data baru. Tetapi dengan hanya 41 pasien positif, satu pasien saja menggeser recall ±2,4 poin, sehingga angka ini kurang presisi. AUC uji (0,847) lebih rendah daripada AUC CV (0,906); selisih wajar untuk sampel kecil dan menunjukkan estimasi CV agak optimistis.
4. **Metrik terpenting pada skrining penyakit jantung: recall (sensitivitas).** *False negative* berarti pasien sakit dipulangkan sebagai sehat (di sini 4 pasien), risikonya fatal; *false positive* hanya berujung pemeriksaan lanjutan. Harganya terlihat jelas: 15 dari 35 orang sehat (43%) ikut dirujuk dan presisi hanya 71%. Recall tidak boleh dibaca sendirian (menandai semua orang sakit memberi recall 100%), maka dilaporkan bersama presisi/spesifisitas dan AUC sebagai ukuran daya pisah model.